# S6 — Grad-CAM for ViT-B/16

Four seed-0 ViT-B/16 models with saved weights, then Grad-CAM with the corrected layer.

**How to run (short notebook, about 1 h; safe even with Run All in the editor)**
1. Settings → **Accelerator: GPU T4 x2**, **Internet: On**. Inputs: SIPaKMeD, Herlev, CRIC, APCData_YOLO.
2. **Run All** (or Save Version → Save & Run All). Keep the tab open and the computer awake.
3. When the last cell finishes: download `results_S6.zip` (right panel → Output → /kaggle/working) **and** File → Download Notebook. Send both.
4. If it stops early, still send the downloaded notebook: every finished run is printed in it as a `RESULT` line.

In [ ]:
%%writefile /kaggle/working/gapfix.py
#!/usr/bin/env python3
"""
gapfix.py (v2) -- multi-dataset cross-domain audit and gap-closing experiments for
single-cell cervical cytology classification.

Datasets: SIPaKMeD, Herlev (single-cell crops shipped with the data) and any number of
point-annotated datasets such as CRIC Cervix and APCData (cells are cut out around the
annotated nucleus centre by the `crops` command).

Subcommands, in order:
  crops      cut fixed-size cell crops around nucleus points (CRIC, APCData, ...)
  manifest   one table of every cell: path, dataset, sub-class, source-image group, split
             (splits are grouped by source image, so cells from one image never leak
             between train and test)
  probe      shortcut probe: can crop size / colour / sharpness alone predict the label?
  ruler      nucleus size of normal cells per dataset -> common physical scale
  train      train on one dataset (or several, comma-separated = leave-one-dataset-out),
             evaluate on every dataset; optional AdaBN and few-shot target fine-tuning
  summarize  cross-dataset matrices, gap tables, per-sub-class errors, adaptation curves

Conditions (one factor changed at a time):
  base        original labels, crop stretched to 224x224, flip + rotate
  harm        harmonised labels: drop Metaplastic, Herlev columnar and ASC-US cells
  harm_gray   harm + grayscale input                                   [diagnostic]
  harm_scale  harm + common physical scale (ruler)                     [diagnostic + fix]
  harm_stain  harm + stain/colour/blur augmentation                    [fix]
  full        harm + scale + stain augmentation                        [proposed fix]
  fix_only    original labels + scale + stain augmentation             [fix, labels unchanged]
"""
import argparse
import copy
import glob
import json
import math
import os
import random
import re
import sys
import time
import warnings

import numpy as np
import pandas as pd
from PIL import Image

# ----------------------------------------------------------------------------------
# Dataset definitions
# ----------------------------------------------------------------------------------
SIPAK_SUB = [("superficial", "Superficial-Intermediate"), ("parabasal", "Parabasal"),
             ("koilocyt", "Koilocytotic"), ("dyskerat", "Dyskeratotic"),
             ("metaplas", "Metaplastic")]
HERLEV_SUB = [("superfic", "normal_superficial"), ("intermed", "normal_intermediate"),
              ("columnar", "normal_columnar"), ("light", "light_dysplastic"),
              ("moderate", "moderate_dysplastic"), ("severe", "severe_dysplastic"),
              ("carcinoma", "carcinoma_in_situ")]
BETHESDA = ["NILM", "ASC-US", "LSIL", "ASC-H", "HSIL", "SCC"]
EXPECTED = {"SIPaKMeD": {"Superficial-Intermediate": 831, "Parabasal": 787,
                         "Koilocytotic": 825, "Dyskeratotic": 813, "Metaplastic": 793},
            "Herlev": {"normal_superficial": 74, "normal_intermediate": 70,
                       "normal_columnar": 98, "light_dysplastic": 182,
                       "moderate_dysplastic": 146, "severe_dysplastic": 197,
                       "carcinoma_in_situ": 150},
            "CRIC": {"NILM": 6779, "ASC-US": 606, "LSIL": 1360, "ASC-H": 925, "HSIL": 1703,
                     "SCC": 161},
            "APCData": {"NILM": 2114, "ASC-US": 333, "LSIL": 444, "ASC-H": 182,
                        "HSIL": 421, "SCC": 125}}
NORMAL_SUBS = {"Superficial-Intermediate", "Parabasal", "normal_superficial",
               "normal_intermediate", "normal_columnar", "NILM"}
# normal cells used as a biological ruler (mostly superficial/intermediate squamous cells)
RULER_SUBS = ["Superficial-Intermediate", "normal_superficial", "normal_intermediate", "NILM"]
IMG_EXT = (".bmp", ".png", ".jpg", ".jpeg", ".tif", ".tiff")

CONDS = {
    "base":       dict(meta="abnormal", columnar="normal", ascus="abnormal", geom="stretch", color="none"),
    "harm":       dict(meta="drop", columnar="drop", ascus="drop", geom="stretch", color="none"),
    "harm_gray":  dict(meta="drop", columnar="drop", ascus="drop", geom="stretch", color="gray"),
    "harm_scale": dict(meta="drop", columnar="drop", ascus="drop", geom="scale", color="none"),
    "harm_stain": dict(meta="drop", columnar="drop", ascus="drop", geom="stretch", color="stainaug"),
    "full":       dict(meta="drop", columnar="drop", ascus="drop", geom="scale", color="stainaug"),
    "fix_only":   dict(meta="abnormal", columnar="normal", ascus="abnormal", geom="scale", color="stainaug"),
}
DROPPED_IN_HARM = {"Metaplastic", "normal_columnar", "ASC-US"}
TIMM_NAMES = {"effnet_b0": "efficientnet_b0", "resnet50": "resnet50",
              "densenet121": "densenet121", "convnext_t": "convnext_tiny",
              "vit_b16": "vit_base_patch16_224", "swin_t": "swin_tiny_patch4_window7_224",
              "mobilenet_v3": "mobilenetv3_large_100"}


HYBRIDS = ("dn121_mamba_kan",)


def make_hybrid(pretrained=True, num_classes=2):
    """DenseNet121 -> 7x7x1024 feature map as 49 tokens -> Mamba (selective state-space) block with residual
    and LayerNorm -> mean pool -> KAN head (1024-256-classes). Pure PyTorch (no mamba-ssm / CUDA build needed),
    exposes the timm-style API used by DANN (num_features, forward_features, forward_head, get_classifier)."""
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import timm

    class MambaBlock(nn.Module):
        def __init__(self, d, d_state=16, expand=2, d_conv=4):
            super().__init__()
            di = expand * d
            self.dt_rank = math.ceil(d / 16)
            self.in_proj = nn.Linear(d, 2 * di, bias=False)
            self.conv = nn.Conv1d(di, di, d_conv, groups=di, padding=d_conv - 1)
            self.x_proj = nn.Linear(di, self.dt_rank + 2 * d_state, bias=False)
            self.dt_proj = nn.Linear(self.dt_rank, di)
            dt = torch.exp(torch.rand(di) * (math.log(0.1) - math.log(0.001)) + math.log(0.001))
            with torch.no_grad():
                self.dt_proj.bias.copy_(dt + torch.log(-torch.expm1(-dt)))
            self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1).float()).repeat(di, 1))
            self.D = nn.Parameter(torch.ones(di))
            self.out_proj = nn.Linear(di, d, bias=False)
            self.d_state = d_state

        def forward(self, x):                      # x: (B, L, d)
            B, L, _ = x.shape
            xz = self.in_proj(x)
            u, z = xz.chunk(2, dim=-1)
            u = F.silu(self.conv(u.transpose(1, 2))[..., :L].transpose(1, 2))
            dbc = self.x_proj(u)
            dt, Bm, Cm = torch.split(dbc, [self.dt_rank, self.d_state, self.d_state], dim=-1)
            dt = F.softplus(self.dt_proj(dt))      # (B, L, di)
            A = -torch.exp(self.A_log.float())     # (di, N)
            dA = torch.exp(dt.unsqueeze(-1) * A)   # (B, L, di, N)
            dBu = dt.unsqueeze(-1) * Bm.unsqueeze(2) * u.unsqueeze(-1)
            h = torch.zeros(B, u.shape[-1], self.d_state, device=x.device, dtype=dA.dtype)
            ys = []
            for t in range(L):                     # 49 tokens: a plain sequential scan is cheap
                h = dA[:, t] * h + dBu[:, t]
                ys.append((h * Cm[:, t].unsqueeze(1)).sum(-1))
            y = torch.stack(ys, 1) + u * self.D
            return self.out_proj(y * F.silu(z))

    class KANLinear(nn.Module):
        """Efficient-KAN layer: SiLU base branch + learnable cubic B-spline on a fixed grid per input/output pair."""
        def __init__(self, i, o, grid=5, k=3, lo=-2.0, hi=2.0):
            super().__init__()
            h = (hi - lo) / grid
            g = torch.arange(-k, grid + k + 1).float() * h + lo
            self.register_buffer("grid", g.expand(i, -1).contiguous())
            self.k = k
            self.base = nn.Linear(i, o)
            self.spline_w = nn.Parameter(torch.randn(o, i, grid + k) * 0.1 / math.sqrt(i))

        def bsplines(self, x):                     # x: (B, i) -> (B, i, grid + k)
            g = self.grid
            x = x.unsqueeze(-1)
            b = ((x >= g[:, :-1]) & (x < g[:, 1:])).to(x.dtype)
            for p in range(1, self.k + 1):
                b = (x - g[:, :-(p + 1)]) / (g[:, p:-1] - g[:, :-(p + 1)]) * b[..., :-1] + \
                    (g[:, p + 1:] - x) / (g[:, p + 1:] - g[:, 1:-p]) * b[..., 1:]
            return b

        def forward(self, x):
            return self.base(F.silu(x)) + torch.einsum("bik,oik->bo", self.bsplines(x), self.spline_w)

    class DMK(nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = timm.create_model("densenet121", pretrained=pretrained, num_classes=0)
            d = self.backbone.num_features          # 1024
            self.num_features = d
            self.mamba = MambaBlock(d)
            self.norm = nn.LayerNorm(d)
            self.head = nn.Sequential(nn.LayerNorm(d), KANLinear(d, 256), KANLinear(256, num_classes))
            self.gradcam_layer = self.backbone.features if hasattr(self.backbone, "features") else self.backbone

        def forward_features(self, x):
            f = self.backbone.forward_features(x)   # (B, 1024, 7, 7)
            t = f.flatten(2).transpose(1, 2).float() if f.dim() == 4 else f.float()
            return self.norm(t + self.mamba(t))     # (B, 49, 1024)

        def forward_head(self, t, pre_logits=False):
            p = t.mean(1)
            return p if pre_logits else self.head(p)

        def get_classifier(self):
            return self.head

        def forward(self, x):
            return self.forward_head(self.forward_features(x))

    return DMK()


def binary_label(sub, meta="abnormal", columnar="normal", ascus="abnormal"):
    """0 = Normal, 1 = Abnormal, None = dropped."""
    if sub == "Metaplastic":
        return {"abnormal": 1, "normal": 0, "drop": None}[meta]
    if sub == "normal_columnar":
        return {"normal": 0, "drop": None}[columnar]
    if sub == "ASC-US":
        return {"abnormal": 1, "drop": None}[ascus]
    return 0 if sub in NORMAL_SUBS else 1


def labels_for(man, cfg):
    return [binary_label(s, cfg["meta"], cfg["columnar"], cfg["ascus"]) for s in man.subclass]


def match_sub(relpath, table):
    parts = [p.lower() for p in relpath.replace("\\", "/").split("/")[:-1]]
    for part in reversed(parts):
        for key, name in table:
            if key in part:
                return name
    return None


def bethesda_from_text(s):
    t = re.sub(r"[\s_\-]", "", str(s).lower())
    if "negative" in t or "nilm" in t or t in ("normal", "0"):
        return "NILM"
    if "ascus" in t:
        return "ASC-US"
    if "asch" in t:
        return "ASC-H"
    if "lsil" in t or "lowgrade" in t:
        return "LSIL"
    if "hsil" in t or "highgrade" in t:
        return "HSIL"
    if "scc" in t or "carcinoma" in t:
        return "SCC"
    return None


def load_rgb(path):
    with Image.open(path) as im:
        return im.convert("RGB")


def need_dir(path, what):
    if not path or not os.path.isdir(path):
        base = "/kaggle/input"
        hint = ""
        if os.path.isdir(base):
            tops = sorted(os.listdir(base))
            hint = f"\n  folders in {base}: {tops}"
        sys.exit(f"[error] {what} folder not found: {path!r}{hint}\n  -> fix the path in cell 1")


def list_images(root):
    return sorted(p for p in glob.glob(os.path.join(root, "**", "*"), recursive=True)
                  if os.path.isfile(p) and p.lower().endswith(IMG_EXT))


# ----------------------------------------------------------------------------------
# crops: point-annotated datasets (CRIC, APCData)
# ----------------------------------------------------------------------------------
def _find_col(cols, *cands, exclude=()):
    low = {c: c.lower() for c in cols}
    for cand in cands:
        for c, l in low.items():
            if cand in l and not any(e in l for e in exclude):
                return c
    return None


def _points_from_csv(a):
    files = [a.csv] if os.path.isfile(a.csv) else sorted(
        glob.glob(os.path.join(a.csv, "**", "*.csv"), recursive=True))
    if not files:
        sys.exit(f"[crops] no CSV found at {a.csv}")
    frames = []
    for f in files:
        d = pd.read_csv(f)
        d["_csv"] = os.path.splitext(os.path.basename(f))[0]
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    cols = list(df.columns)
    c_img = a.col_image or _find_col(cols, "image_filename", "filename", "file", "image_name",
                                     "image", exclude=("doi", "id"))
    exact = {c.lower().strip(): c for c in cols}
    c_x = a.col_x or _find_col(cols, "nucleus_x", "x_center", "xcenter", "center_x",
                               "pos_x", "coord_x") or exact.get("x")
    c_y = a.col_y or _find_col(cols, "nucleus_y", "y_center", "ycenter", "center_y",
                               "pos_y", "coord_y") or exact.get("y")
    c_lab = a.col_label or _find_col(cols, "bethesda", "class", "label", "category", "diagnos")
    c_id = a.col_cell or _find_col(cols, "cell_id", "cellid", "id", exclude=("image",))
    print(f"[crops] {len(df)} rows from {len(files)} csv file(s); columns: {cols}")
    print(f"[crops] using image={c_img!r} x={c_x!r} y={c_y!r} label={c_lab!r} cell_id={c_id!r}")
    if c_x is None or c_y is None or c_lab is None:
        sys.exit("[crops] could not detect x/y/label columns - pass --col_x/--col_y/--col_label")
    if c_img is None:
        print("[crops] no image column: assuming one CSV per image named like the image")
        c_img = "_csv"
    return pd.DataFrame(dict(
        stem=df[c_img].astype(str).map(lambda s: os.path.splitext(os.path.basename(s))[0].lower()),
        x=df[c_x].astype(float), y=df[c_y].astype(float), norm=False, raw_label=df[c_lab],
        cid=df[c_id].values if c_id else np.arange(len(df))))


def _yolo_class_names(root, classes_file=None):
    cands = [classes_file] if classes_file else (
        glob.glob(os.path.join(root, "**", "classes.txt"), recursive=True) +
        glob.glob(os.path.join(root, "**", "*.yaml"), recursive=True))
    for f in cands:
        if not f or not os.path.isfile(f):
            continue
        if f.endswith(".txt"):
            names = [l.strip() for l in open(f) if l.strip()]
            if names:
                return names, f
        else:
            txt = open(f).read()
            try:
                import yaml
                n = yaml.safe_load(txt).get("names")
            except Exception:
                m = re.search(r"names:\s*\[(.*?)\]", txt, re.S)
                n = [x.strip(" '\"\n") for x in m.group(1).split(",")] if m else None
            if isinstance(n, dict):
                n = [n[k] for k in sorted(n)]
            if n:
                return list(n), f
    return None, None


def _points_from_yolo(a):
    lab_root = a.labels or a.images
    names, src = _yolo_class_names(lab_root if a.labels else a.images, a.classes)
    if names is None:
        sys.exit("[crops] YOLO class names not found: pass --classes classes.txt (or data.yaml)")
    print(f"[crops] YOLO class names from {src}: {names}")
    txts = [f for f in glob.glob(os.path.join(lab_root, "**", "*.txt"), recursive=True)
            if os.path.basename(f).lower() not in ("classes.txt", "readme.txt", "notes.txt")
            and "readme" not in os.path.basename(f).lower()]
    rows = []
    for f in txts:
        stem = os.path.splitext(os.path.basename(f))[0].lower()
        for i, line in enumerate(open(f)):
            parts = line.split()
            if len(parts) < 5:
                continue
            c, cx, cy = int(float(parts[0])), float(parts[1]), float(parts[2])
            rows.append(dict(stem=stem, x=cx, y=cy, norm=True,
                             raw_label=names[c] if c < len(names) else f"id{c}", cid=i))
    print(f"[crops] {len(rows)} boxes in {len(txts)} YOLO label files")
    if any(".rf." in r["stem"] for r in rows[:200]):
        print("[crops] WARNING file names contain '.rf.' -> this looks like a Roboflow export, which"
              " may be resized and/or contain augmented duplicates. Run `check` first.")
    return pd.DataFrame(rows)


def cmd_crops(a):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    need_dir(a.images, "images")
    df = _points_from_yolo(a) if a.yolo else _points_from_csv(a)
    imgs = {os.path.splitext(os.path.basename(p))[0].lower(): p for p in list_images(a.images)}
    df["subclass"] = df.raw_label.map(bethesda_from_text)
    unk = df[df.subclass.isna()].raw_label.value_counts()
    if len(unk):
        print(f"[crops] WARNING unmapped labels (dropped): {unk.to_dict()}")
    df = df[df.subclass.notna()]
    os.makedirs(a.out, exist_ok=True)
    half = a.size // 2
    rows, missing = [], set()
    for stem, d in df.groupby("stem"):
        p = imgs.get(stem)
        if p is None:
            missing.add(stem)
            continue
        arr = np.asarray(load_rgb(p))
        H, W = arr.shape[:2]
        pad = np.pad(arr, ((half, half), (half, half), (0, 0)), mode="reflect")
        group = stem.split(".rf.")[0]
        for r in d.to_dict("records"):
            x, y = (r["x"] * W, r["y"] * H) if r["norm"] else (r["x"], r["y"])
            x, y = int(round(x)), int(round(y))
            if not (0 <= x < W and 0 <= y < H):
                continue
            crop = pad[y:y + a.size, x:x + a.size]
            sub = r["subclass"]
            od = os.path.join(a.out, sub)
            os.makedirs(od, exist_ok=True)
            fp = os.path.join(od, f"{stem}_{r['cid']}.png")
            Image.fromarray(crop).save(fp)
            rows.append(dict(path=os.path.abspath(fp), subclass=sub, group=group,
                             centre="cell" if a.yolo else "nucleus"))
    idx = pd.DataFrame(rows)
    idx.to_csv(os.path.join(a.out, "crops_index.csv"), index=False)
    if missing:
        print(f"[crops] WARNING {len(missing)} annotated image names not found in --images, e.g. "
              f"{sorted(missing)[:3]}")
    print(f"[crops] wrote {len(idx)} crops from {idx.group.nunique() if len(idx) else 0} images")
    if len(idx):
        print(idx.subclass.value_counts().reindex(BETHESDA).fillna(0).astype(int).to_string())
        fig, axs = plt.subplots(len(BETHESDA), 6, figsize=(9, 1.6 * len(BETHESDA)))
        for i, sub in enumerate(BETHESDA):
            ex = idx[idx.subclass == sub]
            ex = ex.sample(min(6, len(ex)), random_state=0) if len(ex) else ex
            for j in range(6):
                ax = axs[i, j]
                ax.axis("off")
                if j < len(ex):
                    ax.imshow(load_rgb(ex.path.iloc[j]))
                    if j == 0:
                        ax.set_title(sub, fontsize=7, loc="left")
        fig.tight_layout()
        fig.savefig(os.path.join(a.out, "crops_check.png"), dpi=120)
        print(f"[crops] look at {a.out}/crops_check.png: each crop should show ONE whole cell,"
              f" nucleus in the centre. If cells are cut off, rerun with a larger --size.")


# ----------------------------------------------------------------------------------
# check: is a downloaded copy the same as the official release?
# ----------------------------------------------------------------------------------
OFFICIAL = {
    "CRIC": dict(images=400, size=(1376, 1020), cells=11534,
                 classes={"NILM": 6779, "ASC-US": 606, "LSIL": 1360, "ASC-H": 925,
                          "HSIL": 1703, "SCC": 161},
                 source="figshare collection 4960286 / database.cric.com.br"),
    "APCData": dict(images=425, size=(2048, 1532), cells=3619,
                    classes={"NILM": 2114, "ASC-US": 333, "LSIL": 444, "ASC-H": 182,
                             "HSIL": 421, "SCC": 125},
                    source="Mendeley Data ytd568rh3p (v1)"),
}


def cmd_check(a):
    import hashlib
    from collections import Counter
    need_dir(a.root, a.name)
    off = OFFICIAL[a.name]
    print(f"===== {a.name}: checking {a.root}\n(official: {off['source']})")
    allf = [p for p in glob.glob(os.path.join(a.root, "**", "*"), recursive=True) if os.path.isfile(p)]
    dirs = sorted({os.path.relpath(os.path.dirname(p), a.root) for p in allf})
    print(f"folders: {dirs[:15]}{' ...' if len(dirs) > 15 else ''}")
    imgs = [p for p in allf if p.lower().endswith(IMG_EXT)]
    sizes = Counter()
    for p in imgs:
        with Image.open(p) as im:
            sizes[im.size] += 1
    hashes = Counter()
    for p in imgs:
        with open(p, "rb") as f:
            hashes[hashlib.md5(f.read()).hexdigest()] += 1
    dups = sum(n - 1 for n in hashes.values() if n > 1)
    rf = sum(".rf." in os.path.basename(p) for p in allf)
    split_dirs = [d for d in dirs if re.search(r"(^|/)(train|valid|val|test)(/|$)", d)]
    print(f"images: {len(imgs)}  (official {off['images']})")
    print(f"image sizes: {dict(sizes.most_common(5))}  (official {off['size'][0]}x{off['size'][1]})")
    print(f"exact duplicate images: {dups}")
    if rf:
        print(f"Roboflow-renamed files ('.rf.'): {rf}")
    if split_dirs:
        print(f"pre-made split folders found: {split_dirs[:6]}")
    counts = None
    csvs = [p for p in allf if p.lower().endswith(".csv")]
    txts = [p for p in allf if p.lower().endswith(".txt") and "classes" not in os.path.basename(p).lower()
            and "readme" not in os.path.basename(p).lower()]
    if csvs:
        d = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
        print(f"CSV files: {len(csvs)}, rows: {len(d)}, columns: {list(d.columns)}")
        lab = _find_col(list(d.columns), "bethesda", "class", "label")
        if lab:
            counts = d[lab].map(bethesda_from_text).value_counts()
    if counts is None and txts:
        names, src = _yolo_class_names(a.root)
        ids = Counter()
        for f in txts:
            for line in open(f):
                if len(line.split()) >= 5:
                    ids[int(float(line.split()[0]))] += 1
        print(f"YOLO label files: {len(txts)}, boxes: {sum(ids.values())}, class ids: {dict(ids)}")
        print(f"class names ({src}): {names}")
        if names:
            counts = pd.Series({bethesda_from_text(names[k]) if k < len(names) else f"id{k}": v
                                for k, v in ids.items()}).groupby(level=0).sum()
    verdict = []
    if counts is not None:
        print("\ncells per class      found   official")
        for c in BETHESDA:
            print(f"  {c:8s} {int(counts.get(c, 0)):12d} {off['classes'][c]:10d}")
        tot = int(counts.sum())
        print(f"  {'total':8s} {tot:12d} {off['cells']:10d}")
        verdict.append(("cell counts match", tot == off["cells"] and all(
            int(counts.get(c, 0)) == n for c, n in off["classes"].items())))
    verdict += [("image count matches", len(imgs) == off["images"]),
                ("original resolution", sizes.most_common(1)[0][0] == off["size"] if sizes else False),
                ("no duplicate images", dups == 0),
                ("not a Roboflow export", rf == 0)]
    print("\nVERDICT")
    for k, ok in verdict:
        print(f"  [{'OK' if ok else '!!'}] {k}")
    if all(ok for _, ok in verdict):
        print("  -> identical in size and content to the official release")
    else:
        print("  -> differs from the official release: see the !! lines. A resized or augmented copy"
              " changes the scale and adds duplicates; prefer the official download.")


# ----------------------------------------------------------------------------------
# manifest
# ----------------------------------------------------------------------------------
def grouped_split(d, seed):
    """70/15/15 split stratified on sub-class, grouped by source image."""
    from sklearn.model_selection import StratifiedGroupKFold
    n_groups = d.group.nunique()
    split = pd.Series("train", index=d.index)
    if n_groups < 20:
        print(f"[manifest] WARNING only {n_groups} groups; falling back to ungrouped split")
        g = d.index.astype(str)
    else:
        g = d.group.values
    sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=seed)
    for fold, (_, te) in enumerate(sgkf.split(np.zeros(len(d)), d.subclass.values, g)):
        if fold < 3:
            split.iloc[te] = "test"
        elif fold < 6:
            split.iloc[te] = "val"
    return split


def cmd_manifest(a):
    rows = []
    if a.sipakmed_root:
        need_dir(a.sipakmed_root, "SIPaKMeD")
        sp = list_images(a.sipakmed_root)
        crops = [p for p in sp if "cropped" in p.lower().replace("\\", "/").split("/")[-2]]
        if not crops:
            # no CROPPED sub-folders: keep single-cell crops named <cluster>_<cell>, drop the
            # full cluster images (named <cluster>.bmp)
            crops = [p for p in sp if re.match(r"^\d+_\d+$", os.path.splitext(os.path.basename(p))[0])]
            print(f"[manifest] no CROPPED folders in SIPaKMeD root; kept {len(crops)} files named "
                  f"<cluster>_<cell> out of {len(sp)} images")
            if not crops:
                crops = sp
        for p in crops:
            sub = match_sub(os.path.relpath(p, a.sipakmed_root), SIPAK_SUB)
            if sub:
                # crops are named <clusterimage>_<cell>.bmp inside a class folder
                cluster = os.path.basename(p).split("_")[0]
                rows.append(dict(path=p, dataset="SIPaKMeD", subclass=sub, mask="",
                                 group=f"{sub}/{cluster}"))
    if a.herlev_root:
        need_dir(a.herlev_root, "Herlev")
        hp = list_images(a.herlev_root)
        masks = {}
        for p in hp:
            stem = os.path.splitext(os.path.basename(p))[0]
            if stem.lower().endswith("-d"):
                masks[(os.path.dirname(p), stem[:-2].lower())] = p
        for p in hp:
            stem = os.path.splitext(os.path.basename(p))[0]
            if stem.lower().endswith("-d"):
                continue
            sub = match_sub(os.path.relpath(p, a.herlev_root), HERLEV_SUB)
            if sub:
                rows.append(dict(path=p, dataset="Herlev", subclass=sub, group=p,
                                 mask=masks.get((os.path.dirname(p), stem.lower()), "")))
    for spec in a.crops or []:
        name, d = spec.split("=", 1)
        if not os.path.exists(os.path.join(d, "crops_index.csv")):
            sys.exit(f"[manifest] {d}/crops_index.csv missing: the `crops` step for {name} failed")
        idx = pd.read_csv(os.path.join(d, "crops_index.csv"))
        if "centre" not in idx.columns:
            idx["centre"] = "cell"
        for r in idx.itertuples(index=False):
            rows.append(dict(path=r.path, dataset=name, subclass=r.subclass, mask="",
                             group=f"{name}/{r.group}", centre=r.centre))
    man = pd.DataFrame(rows)
    if man.empty:
        sys.exit("[manifest] no images found")
    man["centre"] = man["centre"].fillna("cell") if "centre" in man.columns else "cell"
    man["split"] = ""
    for ds, d in man.groupby("dataset"):
        man.loc[d.index, "split"] = grouped_split(d, a.seed)
    man.to_csv(a.out, index=False)
    print("\n[manifest] cells per sub-class (expected in brackets):")
    for (ds, sub), n in man.groupby(["dataset", "subclass"]).size().items():
        exp = EXPECTED.get(ds, {}).get(sub)
        flag = "" if exp in (None, n) else "   <-- CHECK"
        print(f"  {ds:9s} {sub:26s} {n:6d} ({exp if exp else '-'}){flag}")
    for ds, d in man.groupby("dataset"):
        exp = EXPECTED.get(ds, {}).get("_total")
        if exp:
            print(f"  {ds:9s} total {len(d)} (published {exp})")
    print(man.groupby(["dataset", "split"]).size().unstack())
    hm = man[man.dataset == "Herlev"]
    if len(hm):
        print(f"[manifest] Herlev images with a -d mask: {(hm['mask'] != '').sum()} / {len(hm)}")
    print(f"[manifest] wrote {a.out}")


# ----------------------------------------------------------------------------------
# probe
# ----------------------------------------------------------------------------------
FEATURE_GROUPS = {
    "size":   ["h", "w", "aspect", "log_area"],
    "colour": ["mean_r", "mean_g", "mean_b", "std_r", "std_g", "std_b",
               "bord_r", "bord_g", "bord_b", "sat_mean"],
    "sharpness": ["lap_var"],
}
FEATURE_GROUPS["all"] = sum(FEATURE_GROUPS.values(), [])


def simple_features(img):
    arr = np.asarray(img).astype(np.float32)
    h, w = arr.shape[:2]
    bw = max(1, int(0.05 * min(h, w)))
    border = np.concatenate([arr[:bw].reshape(-1, 3), arr[-bw:].reshape(-1, 3),
                             arr[:, :bw].reshape(-1, 3), arr[:, -bw:].reshape(-1, 3)])
    g = arr.mean(2)
    lap = (-4 * g[1:-1, 1:-1] + g[:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, :-2] + g[1:-1, 2:])
    mx, mn = arr.max(2), arr.min(2)
    sat = np.where(mx > 0, (mx - mn) / np.maximum(mx, 1e-6), 0)
    f = dict(h=h, w=w, aspect=w / h, log_area=math.log(h * w),
             lap_var=float(lap.var()) if lap.size else 0.0, sat_mean=float(sat.mean()))
    for i, c in enumerate("rgb"):
        f[f"mean_{c}"] = float(arr[..., i].mean())
        f[f"std_{c}"] = float(arr[..., i].std())
        f[f"bord_{c}"] = float(border[:, i].mean())
    return f


def bal_acc(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return float(np.mean([np.mean(p[y == c] == c) for c in (0, 1) if np.any(y == c)]))


def cmd_probe(a):
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    man = pd.read_csv(a.manifest, keep_default_na=False)
    feats = pd.DataFrame([simple_features(load_rgb(p)) for p in man.path])
    df = pd.concat([man.reset_index(drop=True), feats], axis=1)
    dsets = sorted(df.dataset.unique())
    out = []
    for scheme in ["base", "harm"]:
        df["label"] = labels_for(df, CONDS[scheme])
        d = df[df.label.notna()].copy()
        d["label"] = d.label.astype(int)
        for src in dsets:
            S = d[d.dataset == src]
            tr, te = S[S.split != "test"], S[S.split == "test"]
            for gname, cols in FEATURE_GROUPS.items():
                for mname, mk in [
                    ("logreg", lambda: make_pipeline(StandardScaler(), LogisticRegression(
                        max_iter=5000, class_weight="balanced"))),
                    ("gboost", lambda: HistGradientBoostingClassifier(
                        max_iter=200, class_weight="balanced", random_state=0))]:
                    m = mk().fit(tr[cols].values, tr.label.values)
                    rec = dict(scheme=scheme, train=src, features=gname, model=mname,
                               **{f"in:{src}": bal_acc(te.label, m.predict(te[cols].values))})
                    for t in dsets:
                        if t != src:
                            T = d[d.dataset == t]
                            rec[f"cross:{t}"] = bal_acc(T.label, m.predict(T[cols].values))
                    out.append(rec)
    res = pd.DataFrame(out)
    xcols = [c for c in res.columns if c.startswith("cross:")]
    res["in_dist"] = [r[f"in:{r['train']}"] for _, r in res.iterrows()]
    res["cross_mean"] = res[xcols].mean(axis=1)
    res = res[["scheme", "train", "features", "model", "in_dist", "cross_mean"] + xcols]
    os.makedirs(a.out, exist_ok=True)
    res.to_csv(os.path.join(a.out, "probe.csv"), index=False)
    df[["path", "dataset", "subclass", "split"] + FEATURE_GROUPS["all"]].to_csv(
        os.path.join(a.out, "probe_features.csv"), index=False)
    pd.set_option("display.width", 200)
    print(res[res.model == "gboost"].round(3).to_string(index=False))
    print(f"[probe] wrote {a.out}/probe.csv")


# ----------------------------------------------------------------------------------
# ruler
# ----------------------------------------------------------------------------------
def _component_near_centre(mask):
    from skimage.measure import label, regionprops
    lab = label(mask)
    if lab.max() == 0:
        return None
    h, w = mask.shape
    best, bestd = None, 1e9
    for r in regionprops(lab):
        if r.area < 8:
            continue
        d = math.hypot(r.centroid[0] - h / 2, r.centroid[1] - w / 2)
        if d < bestd:
            best, bestd = r, d
    return best


def nucleus_seg(img, centre_frac=0.3, min_contrast=0.12):
    """Seeded nucleus segmentation for a single-cell crop.
    1) darkness signal (grey darkness + haematoxylin), heavily smoothed so specks fade;
    2) seed = darkest smoothed point within `centre_frac` of the crop centre;
    3) region grown at half-way between the seed level and the background level;
    4) tightened if the region is implausibly large (merged with cytoplasm)."""
    from scipy import ndimage as ndi
    from skimage.color import rgb2gray, rgb2hed
    from skimage.filters import gaussian
    from skimage.measure import label, regionprops
    a = np.asarray(img.convert("RGB")).astype(np.float32) / 255.0
    h, w = a.shape[:2]
    A = float(h * w)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        hem = rgb2hed(a)[..., 0]
    dark = 1.0 - rgb2gray(a)
    nrm = lambda x: (x - np.percentile(x, 1)) / (np.percentile(x, 99.5) - np.percentile(x, 1) + 1e-6)
    s = 0.6 * nrm(dark) + 0.4 * nrm(hem)
    s1 = gaussian(s, 1.0)
    ss = gaussian(s, max(1.5, min(h, w) / 40))
    yy, xx = np.mgrid[0:h, 0:w]
    centre = np.hypot(yy - h / 2, xx - w / 2) <= centre_frac * min(h, w)
    seed = np.unravel_index(np.argmax(np.where(centre, ss, -np.inf)), ss.shape)
    peak, bg = s1[seed], float(np.median(s1))
    for frac in (0.5, 0.65, 0.8):
        m = s1 > bg + frac * (peak - bg)
        m = ndi.binary_opening(m, iterations=1)
        m = ndi.binary_fill_holes(m)
        lab = label(m)
        k = lab[seed]
        if k == 0:
            continue
        r = [p for p in regionprops(lab) if p.label == k][0]
        fa = r.area / A
        if fa > 0.2 or r.solidity < 0.85:
            continue
        if r.area < 20:
            return None
        # quality: the region must be clearly darker than a ring around it
        reg = lab == k
        ring = ndi.binary_dilation(reg, iterations=max(2, int(0.25 * math.sqrt(r.area)))) & ~reg
        if ring.any() and (s1[reg].mean() - s1[ring].mean()) < min_contrast:
            return None
        return r
    return None


def centre_frac(row):
    """crops cut around an annotated nucleus point (CRIC) have the nucleus exactly at the
    centre, so the search is kept tight; whole-cell crops allow a wider search."""
    c = row.get("centre", "cell") if hasattr(row, "get") else getattr(row, "centre", "cell")
    return 0.1 if c == "nucleus" else 0.3


def nucleus_otsu(img, cfrac=0.3):
    """(area, region) of the nucleus; kept under its old name for the ruler/xai code."""
    r = nucleus_seg(img, centre_frac=cfrac)
    return (float(r.area), r) if r is not None else (float("nan"), None)


def nucleus_from_herlev_mask(img, mask_path):
    seg = np.asarray(load_rgb(mask_path))
    g = np.asarray(img.convert("L"), dtype=np.float32)
    if seg.shape[:2] != g.shape:
        return float("nan"), None
    _, inv, counts = np.unique(seg.reshape(-1, 3), axis=0, return_inverse=True,
                               return_counts=True)
    inv = inv.reshape(g.shape)
    best, best_mean = None, 1e9
    for ci, n in enumerate(counts):
        if n < 0.005 * g.size:
            continue
        m = g[inv == ci].mean()
        if m < best_mean:
            best, best_mean = ci, m
    if best is None:
        return float("nan"), None
    r = _component_near_centre(inv == best)
    return (float(r.area), r) if r is not None else (float("nan"), None)


def cmd_ruler(a):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    man = pd.read_csv(a.manifest, keep_default_na=False)
    rows = []
    for ds, d in man[(man.subclass.isin(RULER_SUBS)) & (man.split == "train")].groupby("dataset"):
        d = d.sample(min(len(d), a.max_cells), random_state=0)
        for _, r in d.iterrows():
            img = load_rgb(r.path)
            cf = centre_frac(r)
            area, _ = nucleus_otsu(img, cf)
            marea = float("nan")
            if ds == "Herlev" and r["mask"]:
                marea, _ = nucleus_from_herlev_mask(img, r["mask"])
            rows.append(dict(path=r.path, dataset=ds, w=img.size[0], h=img.size[1],
                             nuc_area_otsu=area, nuc_area_mask=marea, cf=cf))
    R = pd.DataFrame(rows)
    R["diam_otsu"] = np.sqrt(4 * R.nuc_area_otsu / np.pi)
    med = R.groupby("dataset")[["diam_otsu"]].median()
    print("[ruler] median nucleus diameter of normal squamous training cells (pixels):")
    print(med.round(1))
    hv = R[(R.dataset == "Herlev") & R.nuc_area_mask.notna() & R.nuc_area_otsu.notna()]
    agree = None
    if len(hv) > 5:
        from scipy.stats import spearmanr
        rho = spearmanr(hv.nuc_area_otsu, hv.nuc_area_mask).correlation
        ratio = float(np.median(hv.nuc_area_otsu / hv.nuc_area_mask))
        agree = dict(spearman=float(rho) if np.isfinite(rho) else None,
                     median_ratio_otsu_over_mask=ratio, n=int(len(hv)))
        print(f"[ruler] Herlev check (Otsu vs expert mask): Spearman {rho:.2f}, "
              f"median ratio {ratio:.2f}, n={len(hv)}")
    ref_ds = a.reference if a.reference in med.index else med.index[0]
    ref = med.loc[ref_ds, "diam_otsu"]
    factor = {ds: float(ref / med.loc[ds, "diam_otsu"]) for ds in med.index}
    sizes = []
    for _, r in man.iterrows():
        with Image.open(r.path) as im:
            sizes.append(max(im.size) * factor.get(r.dataset, 1.0))
    canvas = int(math.ceil(np.percentile(sizes, 95)))
    out = dict(reference=ref_ds, factor=factor, canvas=canvas,
               median_nucleus_diameter_px=med.diam_otsu.round(2).to_dict(),
               herlev_mask_check=agree)
    os.makedirs(a.out, exist_ok=True)
    with open(os.path.join(a.out, "ruler.json"), "w") as f:
        json.dump(out, f, indent=2)
    R.to_csv(os.path.join(a.out, "ruler_cells.csv"), index=False)
    dsets = list(med.index)
    fig, axs = plt.subplots(len(dsets), 6, figsize=(12, 2.2 * len(dsets)), squeeze=False)
    for i, ds in enumerate(dsets):
        ex = R[R.dataset == ds].sample(min(6, (R.dataset == ds).sum()), random_state=0)
        for j in range(6):
            ax = axs[i, j]
            ax.axis("off")
            if j >= len(ex):
                continue
            r = ex.iloc[j]
            img = load_rgb(r.path)
            _, reg = nucleus_otsu(img, r.cf)
            ax.imshow(img)
            if reg is not None:
                minr, minc, maxr, maxc = reg.bbox
                ax.add_patch(plt.Rectangle((minc, minr), maxc - minc, maxr - minr,
                                           fill=False, ec="lime", lw=1.2))
            ax.set_title(f"{ds}  d={r.diam_otsu:.0f}px", fontsize=7)
    fig.tight_layout()
    fig.savefig(os.path.join(a.out, "ruler_check.png"), dpi=130)
    print(f"[ruler] scale factors {json.dumps({k: round(v, 3) for k, v in factor.items()})}; "
          f"canvas {canvas}px -> {a.out}/ruler.json")
    print("[ruler] open ruler_check.png: the green boxes must sit on nuclei")


def to_canvas(img, f, canvas, mask=False):
    """Rescale by the dataset factor, centre on a fixed square canvas (pad with the crop's
    median border colour; centre-crop if larger). After the final uniform resize every
    dataset is shown at the same physical scale and nuclear size is preserved.
    mask=True: the same geometry for a single-channel mask (nearest neighbour, pad 0)."""
    w, h = img.size
    nw, nh = max(1, round(w * f)), max(1, round(h * f))
    img = img.resize((nw, nh), Image.NEAREST if mask else Image.BICUBIC)
    if mask:
        fill = 0
    else:
        arr = np.asarray(img)
        border = np.concatenate([arr[0], arr[-1], arr[:, 0], arr[:, -1]])
        fill = tuple(int(x) for x in np.median(border, axis=0))
    if nw > canvas or nh > canvas:
        l, t = max(0, (nw - canvas) // 2), max(0, (nh - canvas) // 2)
        img = img.crop((l, t, l + min(nw, canvas), t + min(nh, canvas)))
    out = Image.new("L" if mask else "RGB", (canvas, canvas), fill)
    out.paste(img, ((canvas - img.size[0]) // 2, (canvas - img.size[1]) // 2))
    return out


# ----------------------------------------------------------------------------------
# metrics
# ----------------------------------------------------------------------------------
def softmax_np(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def ece_score(y, prob1, bins=10):
    conf = np.maximum(prob1, 1 - prob1)
    acc = ((prob1 >= 0.5).astype(int) == y).astype(float)
    e = 0.0
    for lo, hi in zip(np.linspace(0, 1, bins + 1)[:-1], np.linspace(0, 1, bins + 1)[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)


def metrics_from_logits(y, logits, T=1.0):
    from sklearn.metrics import f1_score, roc_auc_score
    y = np.asarray(y).astype(int)
    p = softmax_np(np.asarray(logits, dtype=np.float64) / T)[:, 1]
    pred = (p >= 0.5).astype(int)
    nr = float(np.mean(pred[y == 0] == 0)) if (y == 0).any() else float("nan")
    ar = float(np.mean(pred[y == 1] == 1)) if (y == 1).any() else float("nan")
    try:
        auc = float(roc_auc_score(y, p))
    except ValueError:
        auc = float("nan")
    return dict(acc=float(np.mean(pred == y)), bal_acc=float(np.nanmean([nr, ar])),
                normal_recall=nr, abnormal_recall=ar,
                f1_macro=float(f1_score(y, pred, average="macro", zero_division=0)),
                auc=auc, ece=ece_score(y, p), n=int(len(y)))


def fit_temperature(y, logits):
    y = np.asarray(y).astype(int)
    z = np.asarray(logits, dtype=np.float64)
    best_t, best_nll = 1.0, 1e18
    for t in np.exp(np.linspace(np.log(0.05), np.log(10.0), 400)):
        p = softmax_np(z / t)
        nll = -np.mean(np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)))
        if nll < best_nll:
            best_t, best_nll = float(t), nll
    return best_t


def save_run(out_dir, run_id, meta, rows, preds):
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, run_id + ".json"), "w") as f:
        json.dump(dict(meta=meta, rows=rows), f, indent=1)
    preds.to_csv(os.path.join(out_dir, run_id + "_preds.csv.gz"), index=False)


# ----------------------------------------------------------------------------------
# train (needs torch + timm: run on a Kaggle GPU)
# ----------------------------------------------------------------------------------
CACHE = {}


def _worker_init(wid):
    import torch
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


class HEDJitter:
    def __init__(self, sigma=0.05, p=0.8):
        self.sigma, self.p = sigma, p

    def __call__(self, img):
        if random.random() > self.p:
            return img
        from skimage.color import hed2rgb, rgb2hed
        hed = rgb2hed(np.asarray(img).astype(np.float32) / 255.0)
        s = self.sigma
        hed = hed * np.random.uniform(1 - s, 1 + s, 3) + np.random.uniform(-s, s, 3)
        return Image.fromarray((np.clip(hed2rgb(hed), 0, 1) * 255).astype(np.uint8))


def build_transforms(cfg, mean, std):
    import torchvision.transforms as T
    train = [T.Resize((224, 224)), T.RandomHorizontalFlip(), T.RandomRotation(15)]
    ev = [T.Resize((224, 224))]
    if cfg["color"] == "gray":
        train.append(T.Grayscale(3))
        ev.append(T.Grayscale(3))
    elif cfg["color"] == "stainaug":
        train += [HEDJitter(0.05, p=0.8),
                  T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.4, hue=0.08),
                  T.RandomGrayscale(p=0.2),
                  T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.3)]
    tail = [T.ToTensor(), T.Normalize(mean, std)]
    return T.Compose(train + tail), T.Compose(ev + tail)


def cmd_train(a):
    import torch
    import torch.nn as nn
    import timm
    from torch.utils.data import DataLoader, Dataset

    cfg = dict(CONDS[a.cond])
    for k in ("meta", "columnar", "ascus"):
        if getattr(a, k):
            cfg[k] = getattr(a, k)
    sources = [s.strip() for s in a.train_ds.split(",") if s.strip()]
    if a.dann_target and (len(sources) != 1 or a.dann_target in sources or a.arch == "custom"):
        sys.exit("[train] --dann_target needs one training dataset, a different target, and a timm model")
    run_id = f"{a.arch}_{'+'.join(sources)}_{a.cond}" + (f"_dann-{a.dann_target}" if a.dann_target else "") + f"_s{a.seed}"
    if os.path.exists(os.path.join(a.out, run_id + ".json")) and not a.overwrite:
        print(f"[train] {run_id} exists - skipping")
        return
    t0 = time.time()
    random.seed(a.seed)
    np.random.seed(a.seed)
    torch.manual_seed(a.seed)
    torch.cuda.manual_seed_all(a.seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    use_amp = dev == "cuda"

    man = pd.read_csv(a.manifest, keep_default_na=False)
    if a.datasets:
        man = man[man.dataset.isin(a.datasets.split(","))]
    man["label"] = labels_for(man, cfg)
    man = man[man.label.notna()].copy()
    man["label"] = man.label.astype(int)
    for s in sources:
        if s not in set(man.dataset):
            sys.exit(f"[train] training dataset {s} not in manifest")
    if a.limit:
        keep = []
        for _, d in man.groupby(["dataset", "split"]):
            keep += list(d.sample(min(len(d), a.limit), random_state=0).index)
        man = man.loc[sorted(keep)]
    targets = sorted(set(man.dataset) - set(sources))
    ruler = None
    if cfg["geom"] == "scale":
        with open(a.ruler) as f:
            ruler = json.load(f)
        miss = set(man.dataset) - set(ruler["factor"])
        if miss:
            sys.exit(f"[train] ruler.json has no scale factor for {miss}; rerun `ruler`")
    for p in man.path:
        if p not in CACHE:
            CACHE[p] = load_rgb(p)

    def new_model():
        if a.arch == "custom":
            import importlib.util
            spec = importlib.util.spec_from_file_location("custom_model", a.custom_module)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            return mod.build_model(num_classes=2)
        if a.arch in HYBRIDS:
            return make_hybrid(pretrained=True)
        return timm.create_model(TIMM_NAMES[a.arch], pretrained=True, num_classes=2)

    model = new_model().to(dev)
    if a.arch in HYBRIDS:
        use_amp = False   # the state-space scan is trained in full precision for numerical stability
    if a.arch == "custom" or a.arch in HYBRIDS:
        mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
    else:
        dc = timm.data.resolve_data_config({}, model=model)
        mean, std = dc["mean"], dc["std"]
    tf_train, tf_eval = build_transforms(cfg, mean, std)

    class CellDS(Dataset):
        def __init__(self, df, tf):
            self.p, self.ds = df.path.values, df.dataset.values
            self.y, self.tf = df.label.values.astype(int), tf

        def __len__(self):
            return len(self.p)

        def __getitem__(self, i):
            img = CACHE[self.p[i]]
            if ruler is not None:
                img = to_canvas(img, ruler["factor"][self.ds[i]], ruler["canvas"])
            return self.tf(img), self.y[i]

    def loader(df, train, bs=None):
        g = torch.Generator()
        g.manual_seed(a.seed)
        return DataLoader(CellDS(df, tf_train if train else tf_eval), batch_size=bs or a.bs,
                          shuffle=train, num_workers=a.workers, worker_init_fn=_worker_init,
                          generator=g, pin_memory=(dev == "cuda"))

    def predict(m, df):
        m.eval()
        out = []
        with torch.no_grad():
            for x, _ in loader(df, False, bs=2 * a.bs):
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                    out.append(m(x.to(dev, non_blocking=True)).float().cpu().numpy())
        return np.concatenate(out)

    def make_scaler():
        try:
            return torch.amp.GradScaler("cuda", enabled=use_amp)
        except (AttributeError, TypeError):
            return torch.cuda.amp.GradScaler(enabled=use_amp)

    def fit(m, df, epochs, lr, class_weight=True, val_df=None):
        ytr = df.label.values
        if class_weight:
            cnt = np.bincount(ytr, minlength=2).astype(float)
            w = torch.tensor(len(ytr) / (2 * np.maximum(cnt, 1)), dtype=torch.float32)
            crit = nn.CrossEntropyLoss(weight=w.to(dev))
        else:
            crit = nn.CrossEntropyLoss()
        opt = torch.optim.Adam(m.parameters(), lr=lr)
        scaler = make_scaler()
        best, best_ba, best_ep, hist = None, -1, 0, []
        dl = loader(df, True, bs=min(a.bs, max(2, len(df))))
        for ep in range(1, epochs + 1):
            m.train()
            tot = 0.0
            for x, y in dl:
                x, y = x.to(dev, non_blocking=True), y.to(dev)
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                    loss = crit(m(x), y)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                tot += loss.item() * len(y)
            rec = dict(epoch=ep, loss=tot / len(df))
            if val_df is not None:
                rec["val_bal_acc"] = metrics_from_logits(val_df.label.values,
                                                         predict(m, val_df))["bal_acc"]
                if rec["val_bal_acc"] > best_ba:
                    best_ba, best_ep = rec["val_bal_acc"], ep
                    best = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
            hist.append(rec)
            print(f"  ep{ep:02d} loss {rec['loss']:.4f}" +
                  (f" val_bal_acc {rec['val_bal_acc']:.4f}" if val_df is not None else ""),
                  flush=True)
        if best is not None:
            m.load_state_dict(best)
        return hist, best_ep

    S = man[man.dataset.isin(sources)]
    S_tr, S_va = S[S.split == "train"], S[S.split == "val"]
    print(f"[train] {run_id}: cfg={cfg} train={len(S_tr)} val={len(S_va)} "
          f"targets={targets} device={dev}", flush=True)
    lr = a.lr or (3e-5 if a.arch in ("vit_b16", "swin_t") else 1e-4)
    ft_lr = a.ft_lr or lr

    def fit_dann(m, df, tdf, epochs, lr, val_df):
        """Domain-adversarial training (Ganin et al., 2016): a gradient-reversal layer feeds the
        pooled backbone features to a domain classifier (source vs. unlabelled target images);
        lambda follows 2/(1+exp(-10p))-1. Model selection uses source validation data only."""
        class GRL(torch.autograd.Function):
            @staticmethod
            def forward(ctx, x, lam):
                ctx.lam = lam
                return x.view_as(x)

            @staticmethod
            def backward(ctx, g):
                return -ctx.lam * g, None
        dom = nn.Sequential(nn.Linear(m.num_features, 256), nn.ReLU(), nn.Linear(256, 2)).to(dev)
        cnt = np.bincount(df.label.values, minlength=2).astype(float)
        w = torch.tensor(len(df) / (2 * np.maximum(cnt, 1)), dtype=torch.float32).to(dev)
        crit, dcrit = nn.CrossEntropyLoss(weight=w), nn.CrossEntropyLoss()
        opt = torch.optim.Adam(list(m.parameters()) + list(dom.parameters()), lr=lr)
        scaler = make_scaler()
        dl_s, dl_t = loader(df, True), loader(tdf, True)
        it_t = iter(dl_t)
        steps, best, best_ba, best_ep, hist = len(dl_s), None, -1, 0, []
        for ep in range(1, epochs + 1):
            m.train()
            dom.train()
            tot = tot_d = 0.0
            for i, (xs, ys) in enumerate(dl_s):
                try:
                    xt, _ = next(it_t)
                except StopIteration:
                    it_t = iter(dl_t)
                    xt, _ = next(it_t)
                prog = ((ep - 1) * steps + i) / (epochs * steps)
                lam = 2.0 / (1.0 + math.exp(-10 * prog)) - 1.0
                xs, ys, xt = xs.to(dev), ys.to(dev), xt.to(dev)
                ns = len(xs)
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                    f = m.forward_head(m.forward_features(torch.cat([xs, xt])), pre_logits=True)
                    logits = m.get_classifier()(f[:ns])
                    dlog = dom(GRL.apply(f, lam))
                    dy = torch.cat([torch.zeros(ns), torch.ones(len(xt))]).long().to(dev)
                    lc, ld = crit(logits, ys), dcrit(dlog, dy)
                    loss = lc + ld
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                tot += lc.item() * ns
                tot_d += ld.item() * ns
            vb = metrics_from_logits(val_df.label.values, predict(m, val_df))["bal_acc"]
            hist.append(dict(epoch=ep, loss=tot / len(df), domain_loss=tot_d / len(df), val_bal_acc=vb, lam=lam))
            print(f"  ep{ep:02d} cls {tot / len(df):.4f} dom {tot_d / len(df):.4f} val_bal_acc {vb:.4f}", flush=True)
            if vb > best_ba:
                best_ba, best_ep = vb, ep
                best = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
        m.load_state_dict(best)
        return hist, best_ep

    if a.dann_target:
        Tg_d = man[man.dataset == a.dann_target]
        hist, best_ep = fit_dann(model, S_tr, Tg_d[Tg_d.split == "train"], a.epochs, lr, S_va)
    else:
        hist, best_ep = fit(model, S_tr, a.epochs, lr, True, S_va)
    Tval = fit_temperature(S_va.label.values, predict(model, S_va))
    if a.save_weights:
        os.makedirs(a.out, exist_ok=True)
        torch.save(model.state_dict(), os.path.join(a.out, run_id + ".pt"))

    rows, preds = [], []

    def evaluate(m, method, ev, test_ds, df, calib_T=None):
        z = predict(m, df)
        for calib, t in [("raw", 1.0)] + ([("TS", calib_T)] if calib_T else []):
            r = dict(method=method, eval=ev, test_ds=test_ds, calib=calib, T=t)
            r.update(metrics_from_logits(df.label.values, z, t))
            rows.append(r)
        preds.append(pd.DataFrame(dict(method=method, eval=ev, test_ds=test_ds,
                                       path=df.path.values, subclass=df.subclass.values,
                                       label=df.label.values, logit0=z[:, 0], logit1=z[:, 1])))
        return rows[-1]

    if a.dann_target:
        # DANN results are reported only on its own target (the target's test split was never seen)
        for s in sources:
            evaluate(model, "dann", "in_dist", s, S[(S.dataset == s) & (S.split == "test")], Tval)
        Tg = man[man.dataset == a.dann_target]
        evaluate(model, "dann", "cross_full", a.dann_target, Tg, Tval)
        r = evaluate(model, "dann", "cross_tgtTest", a.dann_target, Tg[Tg.split == "test"], Tval)
        print(f"[train] DANN -> {a.dann_target}: target-test bal_acc {r['bal_acc']:.3f}", flush=True)
        targets = []
    for s in (sources if not a.dann_target else []):
        r = evaluate(model, "src", "in_dist", s, S[(S.dataset == s) & (S.split == "test")], Tval)
        print(f"[train] in-dist {s}: bal_acc {r['bal_acc']:.3f}", flush=True)
    for t in targets:
        Tg = man[man.dataset == t]
        evaluate(model, "src", "cross_full", t, Tg, Tval)
        r = evaluate(model, "src", "cross_tgtTest", t, Tg[Tg.split == "test"], Tval)
        print(f"[train] cross {t}: bal_acc {r['bal_acc']:.3f}", flush=True)

    adapt_to = targets if a.adapt_to == "all" else [t for t in a.adapt_to.split(",") if t in targets]
    if a.dann_target:
        adapt_to = []
    shots = [int(x) for x in a.fewshot.split(",") if x.strip()]
    has_bn = any(isinstance(mod, nn.modules.batchnorm._BatchNorm) for mod in model.modules())
    for t in adapt_to:
        Tg = man[man.dataset == t]
        T_tr, T_te = Tg[Tg.split == "train"], Tg[Tg.split == "test"]
        if a.adabn and has_bn:
            m2 = copy.deepcopy(model)
            m2.eval()
            for mod in m2.modules():
                if isinstance(mod, nn.modules.batchnorm._BatchNorm):
                    mod.reset_running_stats()
                    mod.momentum = None
                    mod.train()
            with torch.no_grad():
                for x, _ in loader(T_tr, False, bs=2 * a.bs):
                    m2(x.to(dev))
            evaluate(m2, "adabn", "cross_tgtTest", t, T_te)
            del m2
        for k in shots:
            rng = np.random.RandomState(a.seed * 1000 + k)
            idx = []
            for c in (0, 1):
                pool = T_tr.index[T_tr.label == c].values
                idx += list(rng.choice(pool, size=min(k, len(pool)), replace=False))
            shot = T_tr.loc[idx]
            for init in (["src", "imagenet"] if a.ft_scratch else ["src"]):
                m3 = copy.deepcopy(model) if init == "src" else new_model().to(dev)
                fit(m3, shot, a.ft_epochs, ft_lr, class_weight=False)
                tag = f"ft{k}" if init == "src" else f"ftscratch{k}"
                evaluate(m3, tag, "cross_tgtTest", t, T_te)
                if init == "src":
                    for s in sources:
                        evaluate(m3, tag, "in_dist", s, S[(S.dataset == s) & (S.split == "test")])
                del m3
                if dev == "cuda":
                    torch.cuda.empty_cache()
        print(f"[train] adaptation to {t} done", flush=True)

    meta = dict(run_id=run_id, arch=a.arch, train="+".join(sources), cond=a.cond, seed=a.seed,
                dann_target=a.dann_target or "",
                cfg=cfg, T=Tval, best_ep=best_ep, hist=hist, epochs=a.epochs, lr=lr,
                minutes=(time.time() - t0) / 60, limit=a.limit)
    save_run(a.out, run_id, meta, rows, pd.concat(preds, ignore_index=True))
    print(f"[train] saved {run_id} in {(time.time() - t0) / 60:.1f} min", flush=True)


# ----------------------------------------------------------------------------------
# xai: quantitative Grad-CAM against nucleus masks on all datasets
# ----------------------------------------------------------------------------------
def nucleus_mask(img, dataset, mask_path="", cfrac=0.3):
    """Boolean nucleus mask for a cell crop: Herlev expert segmentation when available,
    otherwise the darkest Otsu component closest to the crop centre (CRIC crops are centred
    on the annotated nucleus point)."""
    reg = None
    if dataset == "Herlev" and mask_path:
        _, reg = nucleus_from_herlev_mask(img, mask_path)
    if reg is None:
        _, reg = nucleus_otsu(img, cfrac)
    if reg is None:
        return None
    m = np.zeros((img.size[1], img.size[0]), bool)
    m[reg.coords[:, 0], reg.coords[:, 1]] = True
    frac = m.mean()
    return m if 0.002 <= frac <= 0.6 else None


def _cam_layer(model, arch):
    """(module, kind): kind = 'nchw' feature map, 'nhwc' feature map, or 'tokens'."""
    if arch == "custom" or arch in HYBRIDS:
        if hasattr(model, "gradcam_layer"):
            return model.gradcam_layer, getattr(model, "gradcam_kind", "nchw")
        raise SystemExit("[xai] custom model: define model.gradcam_layer (and gradcam_kind)")
    if arch == "effnet_b0":
        return model.bn2, "nchw"
    if arch == "resnet50":
        return model.layer4, "nchw"
    if arch == "densenet121":
        return model.features, "nchw"
    if arch == "convnext_t":
        return model.stages[-1], "nchw"
    if arch == "mobilenet_v3":
        return model.blocks[-1], "nchw"
    if arch == "vit_b16":
        # the classifier reads only the class token after the last block, so patch tokens at the
        # block OUTPUT receive no gradient; the standard target is the last block's first LayerNorm
        return model.blocks[-1].norm1, "tokens"
    if arch == "swin_t":
        return model.layers[-1], "nhwc"
    raise SystemExit(f"[xai] no Grad-CAM layer defined for {arch}")


def cmd_xai(a):
    import torch
    import torch.nn.functional as F
    import timm
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from scipy.stats import mannwhitneyu, wilcoxon

    cfg = CONDS[a.cond]
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    man = pd.read_csv(a.manifest, keep_default_na=False)
    man["label"] = labels_for(man, cfg)
    man = man[man.label.notna()].copy()
    man["label"] = man.label.astype(int)
    dsets = sorted(man.dataset.unique())
    ruler = None
    if cfg["geom"] == "scale":
        with open(a.ruler) as f:
            ruler = json.load(f)
    os.makedirs(a.out, exist_ok=True)

    # evaluation cells: test split of every dataset, with a usable nucleus mask
    cells = []
    for ds in dsets:
        d = man[(man.dataset == ds) & (man.split == "test")]
        d = d.sample(min(len(d), a.max_images), random_state=0)
        for r in d.itertuples(index=False):
            img = load_rgb(r.path)
            m = nucleus_mask(img, ds, r.mask, centre_frac(r))
            if m is not None:
                cells.append((r, img, m))
    print(f"[xai] {len(cells)} test cells with nucleus masks: "
          f"{pd.Series([c[0].dataset for c in cells]).value_counts().to_dict()}")
    os.makedirs(a.out, exist_ok=True)
    chk = os.path.join(a.out, "xai_mask_check.png")
    if not os.path.exists(chk):
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, axs = plt.subplots(len(dsets), 8, figsize=(12, 1.6 * len(dsets)), squeeze=False)
        for i, ds in enumerate(dsets):
            ex = [c for c in cells if c[0].dataset == ds]
            rng = np.random.RandomState(1)
            pick = rng.choice(len(ex), size=min(8, len(ex)), replace=False) if ex else []
            for j in range(8):
                axs[i, j].axis("off")
            for j, k in enumerate(pick):
                _, img, m = ex[k]
                axs[i, j].imshow(img)
                axs[i, j].contour(m, levels=[0.5], colors="lime", linewidths=0.7)
                if j == 0:
                    axs[i, j].set_title(ds, fontsize=7, loc="left")
        fig.tight_layout()
        fig.savefig(chk, dpi=120)
        plt.close(fig)
        print(f"[xai] mask check sheet: {chk} (green outline should follow the nucleus)")

    def geom(img, ds, is_mask=False):
        if ruler is not None:
            img = to_canvas(img, ruler["factor"][ds], ruler["canvas"], mask=is_mask)
        return img.resize((224, 224), Image.NEAREST if is_mask else Image.BILINEAR)

    rows, examples = [], {}
    ex_paths = {ds: [c[0].path for c in cells if c[0].dataset == ds][:2] for ds in dsets}
    for tr in dsets:
        wpath = os.path.join(a.runs, f"{a.arch}_{tr}_{a.cond}_s{a.seed}.pt")
        if not os.path.exists(wpath):
            print(f"[xai] missing weights {wpath} - skipped")
            continue
        if a.arch == "custom":
            import importlib.util
            spec = importlib.util.spec_from_file_location("custom_model", a.custom_module)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            model = mod.build_model(num_classes=2)
            mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
        elif a.arch in HYBRIDS:
            model = make_hybrid(pretrained=False)
            mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
        else:
            model = timm.create_model(TIMM_NAMES[a.arch], pretrained=False, num_classes=2)
            dc = timm.data.resolve_data_config({}, model=model)
            mean, std = dc["mean"], dc["std"]
        model.load_state_dict(torch.load(wpath, map_location="cpu"))
        model.to(dev).eval()
        layer, kind = _cam_layer(model, a.arch)
        npre = getattr(model, "num_prefix_tokens", 1)
        mean_t = torch.tensor(mean).view(1, 3, 1, 1).to(dev)
        std_t = torch.tensor(std).view(1, 3, 1, 1).to(dev)
        store = {}

        def fwd_hook(_m, _i, o):
            store["a"] = o
            o.register_hook(lambda g: store.__setitem__("g", g))
        h = layer.register_forward_hook(fwd_hook)
        for b0 in range(0, len(cells), a.bs):
            batch = cells[b0:b0 + a.bs]
            ims = []
            for r, img, _ in batch:
                im = geom(img, r.dataset)
                if cfg["color"] == "gray":
                    im = im.convert("L").convert("RGB")
                ims.append(np.asarray(im, dtype=np.float32) / 255.0)
            x = torch.tensor(np.stack(ims)).permute(0, 3, 1, 2).to(dev)
            x = ((x - mean_t) / std_t).requires_grad_(False)
            model.zero_grad(set_to_none=True)
            out = model(x)
            pred = out.argmax(1)
            out.gather(1, pred[:, None]).sum().backward()
            A, G = store["a"].detach(), store["g"].detach()
            if kind == "tokens":
                A, G = A[:, npre:, :], G[:, npre:, :]
                n = int(round(math.sqrt(A.shape[1])))
                A = A.reshape(A.shape[0], n, n, -1).permute(0, 3, 1, 2)
                G = G.reshape(G.shape[0], n, n, -1).permute(0, 3, 1, 2)
            elif kind == "nhwc":
                A, G = A.permute(0, 3, 1, 2), G.permute(0, 3, 1, 2)
            cam = F.relu((G.mean((2, 3), keepdim=True) * A).sum(1, keepdim=True))
            cam = F.interpolate(cam, size=(224, 224), mode="bilinear", align_corners=False)[:, 0]
            cam = cam.cpu().numpy()
            prob = torch.softmax(out.detach().float(), 1)[:, 1].cpu().numpy()
            for (r, img, m), c, pr, pd_ in zip(batch, cam, prob, pred.cpu().numpy()):
                mk = np.asarray(geom(Image.fromarray(m.astype(np.uint8) * 255), r.dataset, True)) > 127
                tot = float(c.sum())
                if tot <= 0 or not mk.any():
                    continue
                energy = float(c[mk].sum() / tot)
                area = float(mk.mean())
                peak = np.unravel_index(np.argmax(c), c.shape)
                rows.append(dict(arch=a.arch, cond=a.cond, seed=a.seed, model_train=tr, eval_ds=r.dataset,
                                 regime="in-domain" if tr == r.dataset else "cross-domain",
                                 path=r.path, subclass=r.subclass, label=int(r.label), pred=int(pd_),
                                 correct=int(pd_) == int(r.label), p_abnormal=float(pr),
                                 energy_in_nucleus=energy, nucleus_area=area,
                                 enrichment=energy / area, pointing_hit=bool(mk[peak])))
                if r.path in ex_paths.get(r.dataset, []):
                    examples[(tr, r.path)] = (np.asarray(geom(img, r.dataset)), mk, c)
        h.remove()
        del model
        if dev == "cuda":
            torch.cuda.empty_cache()
        print(f"[xai] {a.arch} trained on {tr}: done", flush=True)
    if not rows:
        sys.exit("[xai] nothing computed - are the .pt weights in --runs?")
    R = pd.DataFrame(rows)
    xai_report(R, examples, ex_paths, dsets, a)


def xai_report(R, examples, ex_paths, dsets, a):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from scipy.stats import mannwhitneyu, wilcoxon
    tag = f"{a.arch}_{a.cond}_s{a.seed}"
    R.to_csv(os.path.join(a.out, f"xai_per_image_{tag}.csv"), index=False)

    # statistics: same images, in-domain model vs each cross-domain model (paired)
    def holm(ps):
        ps = np.asarray(ps, float)
        order = np.argsort(ps)
        adj = np.empty_like(ps)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (len(ps) - i) * ps[k])
            adj[k] = min(1.0, run)
        return adj
    stats = []
    for ev in dsets:
        ind = R[(R.eval_ds == ev) & (R.regime == "in-domain")].set_index("path")
        if ind.empty:
            continue
        for tr in sorted(set(R.model_train) - {ev}):
            cr = R[(R.eval_ds == ev) & (R.model_train == tr)].set_index("path")
            j = ind.join(cr, lsuffix="_in", rsuffix="_cross", how="inner")
            if len(j) < 10:
                continue
            diff = j.enrichment_cross - j.enrichment_in
            try:
                w = wilcoxon(j.enrichment_cross, j.enrichment_in)
                pv = float(w.pvalue)
            except ValueError:
                pv = float("nan")
            nz = diff[diff != 0]
            r_rb = float((np.sign(nz) * nz.abs().rank()).sum() / nz.abs().rank().sum()) if len(nz) else 0.0
            wrong = cr[~cr.correct].enrichment
            right = cr[cr.correct].enrichment
            p_cw = float(mannwhitneyu(wrong, right).pvalue) if len(wrong) >= 5 and len(right) >= 5 else float("nan")
            stats.append(dict(arch=a.arch, cond=a.cond, eval_ds=ev, cross_model_trained_on=tr, n=len(j),
                              enrich_in_median=float(j.enrichment_in.median()),
                              enrich_cross_median=float(j.enrichment_cross.median()),
                              frac_cells_cross_higher=float((diff > 0).mean()),
                              wilcoxon_p=pv, rank_biserial_r=r_rb,
                              pointing_in=float(j.pointing_hit_in.mean()),
                              pointing_cross=float(j.pointing_hit_cross.mean()),
                              cross_acc=float(cr.correct.mean()),
                              enrich_cross_wrong_median=float(wrong.median()) if len(wrong) else float("nan"),
                              enrich_cross_right_median=float(right.median()) if len(right) else float("nan"),
                              wrong_vs_right_p=p_cw))
    S = pd.DataFrame(stats)
    if len(S):
        S["wilcoxon_p_holm"] = holm(S.wilcoxon_p.fillna(1).values)
        S["wrong_vs_right_p_holm"] = holm(S.wrong_vs_right_p.fillna(1).values)
        S.round(4).to_csv(os.path.join(a.out, f"table_xai_{tag}.csv"), index=False)
        pd.set_option("display.width", 220)
        print(S[["eval_ds", "cross_model_trained_on", "n", "enrich_in_median", "enrich_cross_median",
                 "frac_cells_cross_higher", "wilcoxon_p_holm", "enrich_cross_wrong_median",
                 "enrich_cross_right_median", "wrong_vs_right_p_holm"]].round(3).to_string(index=False))

    # figure 1: median enrichment (IQR) per evaluation dataset, in-domain vs each cross model
    fig, ax = plt.subplots(figsize=(1.6 + 1.4 * len(dsets), 2.8))
    pal = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#6250d6"]
    trains = sorted(R.model_train.unique())
    w = 0.8 / max(1, len(trains))
    for k, tr in enumerate(trains):
        for i, ev in enumerate(dsets):
            v = R[(R.eval_ds == ev) & (R.model_train == tr)].enrichment
            if v.empty:
                continue
            x = i + (k - (len(trains) - 1) / 2) * w
            q1, med, q3 = v.quantile([0.25, 0.5, 0.75])
            ax.bar(x, med, w * 0.9, color=pal[k % len(pal)], edgecolor="#0b0b0b" if tr == ev else "none",
                   lw=1.2, label=f"model trained on {tr}" if i == 0 else None)
            ax.plot([x, x], [q1, q3], color="#0b0b0b", lw=0.8)
    ax.axhline(1, ls="--", color="#52514e", lw=0.8)
    ax.set_xticks(range(len(dsets)))
    ax.set_xticklabels([f"{d} cells" for d in dsets], fontsize=7)
    ax.set_ylabel("Grad-CAM nucleus enrichment\n(median, IQR; 1 = no preference)", fontsize=7)
    ax.set_title(f"{a.arch}: black outline = in-domain model", fontsize=8)
    ax.legend(fontsize=6, ncol=2)
    fig.tight_layout()
    fig.savefig(os.path.join(a.out, f"fig_xai_enrichment_{tag}.png"), dpi=200)
    plt.close(fig)

    # figure 2: examples (rows = evaluation cells, cols = image + one CAM per model)
    ex_rows = [(ds, p) for ds in dsets for p in ex_paths.get(ds, [])]
    if ex_rows and examples:
        fig, axs = plt.subplots(len(ex_rows), 1 + len(trains), figsize=(1.7 * (1 + len(trains)), 1.75 * len(ex_rows)),
                                squeeze=False)
        for i, (ds, pth) in enumerate(ex_rows):
            base = next((examples[(t, pth)] for t in trains if (t, pth) in examples), None)
            for j in range(1 + len(trains)):
                axs[i, j].axis("off")
            if base is None:
                continue
            im, mk, _ = base
            axs[i, 0].imshow(im)
            axs[i, 0].contour(mk, levels=[0.5], colors="lime", linewidths=0.8)
            axs[i, 0].set_title(f"{ds} cell", fontsize=6)
            for j, tr in enumerate(trains, start=1):
                if (tr, pth) not in examples:
                    continue
                im, mk, c = examples[(tr, pth)]
                axs[i, j].imshow(im)
                axs[i, j].imshow(c / (c.max() + 1e-9), cmap="jet", alpha=0.45)
                axs[i, j].contour(mk, levels=[0.5], colors="lime", linewidths=0.6)
                axs[i, j].set_title(f"trained on {tr}" + (" (in)" if tr == ds else ""), fontsize=6)
        fig.tight_layout()
        fig.savefig(os.path.join(a.out, f"fig_xai_examples_{tag}.png"), dpi=170)
        plt.close(fig)
    print(f"[xai] wrote {a.out}/table_xai_{tag}.csv and figures")


# ----------------------------------------------------------------------------------
# summarize
# ----------------------------------------------------------------------------------
def _ms(x):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return ""
    return f"{x.mean():.3f} ± {x.std(ddof=1):.3f}" if len(x) > 1 else f"{x.mean():.3f}"


def _latex(df, path):
    cols = list(df.columns)
    lines = ["\\begin{tabular}{" + "l" * len(cols) + "}", "\\toprule",
             " & ".join(str(c).replace("_", "\\_") for c in cols) + " \\\\", "\\midrule"]
    for _, r in df.iterrows():
        lines.append(" & ".join(str(v).replace("±", "$\\pm$").replace("_", "\\_")
                                for v in r.values) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}"]
    with open(path, "w") as f:
        f.write("\n".join(lines))


def _mkey(m):
    if m == "src":
        return (0, 0)
    if m == "thr_gmm":
        return (1, 0)
    if m == "adabn":
        return (1, 1)
    if m == "dann":
        return (1, 2)
    if m.startswith("thr") and m[3:].isdigit():
        return (2, int(m[3:]))
    if m.startswith("ftscratch"):
        return (4, int(m[9:]))
    if m.startswith("ft") and m[2:].isdigit():
        return (3, int(m[2:]))
    if m == "thr_oracle":
        return (5, 0)
    return (6, 0)


def _best_threshold(m, y):
    """threshold on the logit margin that maximises balanced accuracy on (m, y);
    ties are broken towards the original threshold 0 (smallest change)"""
    u = np.unique(m)
    cand = np.concatenate([[-np.inf], (u[1:] + u[:-1]) / 2 if len(u) > 1 else [], [np.inf], [0.0]])
    pos, neg = (y == 1), (y == 0)
    best, bt = -1, 0.0
    for t in cand:
        p = m >= t
        ba = 0.5 * ((p[pos].mean() if pos.any() else 0) + ((~p)[neg].mean() if neg.any() else 0))
        if ba > best + 1e-12 or (abs(ba - best) <= 1e-12 and abs(t) < abs(bt)):
            best, bt = ba, t
    return float(bt)


def _ba_at(m, y, t):
    p = m >= t
    nr = float((~p)[y == 0].mean()) if (y == 0).any() else float("nan")
    ar = float(p[y == 1].mean()) if (y == 1).any() else float("nan")
    return float(np.nanmean([nr, ar])), nr, ar


def _gmm_threshold(m):
    """label-free: fit a 2-component Gaussian mixture to the unlabelled target margins and
    return the point between the component means where both weighted densities are equal"""
    from sklearn.mixture import GaussianMixture
    if len(m) < 20:
        return 0.0
    g = GaussianMixture(2, random_state=0).fit(m.reshape(-1, 1))
    mu = g.means_.ravel()
    if abs(mu[0] - mu[1]) < 1e-6:
        return 0.0
    lo, hi = sorted(mu)
    grid = np.linspace(lo, hi, 400).reshape(-1, 1)
    post = g.predict_proba(grid)[:, int(np.argmax(mu))]
    return float(grid[np.argmin(np.abs(post - 0.5)), 0])


def threshold_posthoc(P, man, shots, reps=20):
    """Re-set only the decision threshold of a trained source model for a new target dataset:
    thr_gmm (no labels), thr{k} (k labelled target-train cells per class), thr_oracle (best
    possible threshold on the target test split = ceiling of any threshold-only fix).
    Evaluated on the target test split, so directly comparable with AdaBN / ft{k}."""
    split = dict(zip(man.path, man.split))
    X = P[(P.method == "src") & (P["eval"] == "cross_full") & ~P.train.str.contains(r"\+")]
    out = []
    for (rid, td), d in X.groupby(["run_id", "test_ds"]):
        sp = d.path.map(split)
        tr, te = d[sp == "train"], d[sp == "test"]
        if tr.label.nunique() < 2 or te.label.nunique() < 2:
            continue
        mtr = (tr.logit1 - tr.logit0).values
        ytr = tr.label.values.astype(int)
        mte = (te.logit1 - te.logit0).values
        yte = te.label.values.astype(int)
        info = {k: d[k].iloc[0] for k in ("run_id", "arch", "train", "cond", "seed")}

        def add(method, ba, nr, ar, t):
            out.append(dict(info, method=method, eval="cross_tgtTest", test_ds=td, calib="raw",
                            bal_acc=ba, normal_recall=nr, abnormal_recall=ar, threshold=t,
                            n=int(len(te))))
        add("thr_base", *_ba_at(mte, yte, 0.0), 0.0)
        t = _gmm_threshold(mtr)
        add("thr_gmm", *_ba_at(mte, yte, t), t)
        t = _best_threshold(mte, yte)
        add("thr_oracle", *_ba_at(mte, yte, t), t)
        rng = np.random.RandomState(int(info["seed"]) * 7919 + 13)
        for k in shots:
            res = []
            for _ in range(reps):
                idx = np.concatenate([rng.choice(np.where(ytr == c)[0], size=min(k, (ytr == c).sum()),
                                                 replace=False) for c in (0, 1)])
                t = _best_threshold(mtr[idx], ytr[idx])
                res.append(_ba_at(mte, yte, t) + (t,))
            r = np.nanmean(np.array(res, dtype=float), axis=0)
            add(f"thr{k}", *r)
    return pd.DataFrame(out)


def cmd_summarize(a):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.ticker
    os.makedirs(a.out, exist_ok=True)
    rows, preds = [], []
    ids = ("run_id", "arch", "train", "cond", "seed")
    for jf in sorted(glob.glob(os.path.join(a.runs, "*.json"))):
        with open(jf) as f:
            d = json.load(f)
        m = d["meta"]
        for r in d["rows"]:
            rows.append({**{k: m[k] for k in ids}, **r})
        pf = jf[:-5] + "_preds.csv.gz"
        if os.path.exists(pf) and not a.no_preds:
            p = pd.read_csv(pf)
            for k in ids:
                p[k] = m[k]
            p["T_src"] = float(m.get("T", 1.0))
            preds.append(p)
    if not rows:
        sys.exit(f"[summarize] no runs in {a.runs}")
    L = pd.DataFrame(rows)
    L.to_csv(os.path.join(a.out, "all_runs_long.csv"), index=False)
    P = pd.concat(preds, ignore_index=True) if preds else pd.DataFrame()
    raw = L[(L.calib == "raw")]
    src = raw[raw.method == "src"]
    single = src[~src.train.str.contains(r"\+")]
    dsets = sorted(set(L.test_ds))
    cond_order = {c: i for i, c in enumerate(CONDS)}

    try:
        # (1) cross-dataset matrix: rows = train dataset, columns = test dataset
        mat = single[single["eval"].isin(["in_dist", "cross_full"])]
        M = mat.groupby(["arch", "cond", "train", "test_ds"]).bal_acc.apply(_ms).unstack("test_ds")
        M = M.reindex(columns=dsets).reset_index()
        M.to_csv(os.path.join(a.out, "table_matrix_balacc.csv"), index=False)
        Mauc = mat.groupby(["arch", "cond", "train", "test_ds"]).auc.apply(_ms).unstack("test_ds")
        Mauc.reindex(columns=dsets).reset_index().to_csv(
            os.path.join(a.out, "table_matrix_auroc.csv"), index=False)

        # (2) one-line-per-model short result: mean in-dist vs mean cross over all pairs
        per_seed = mat.groupby(["arch", "cond", "seed", "eval"]).bal_acc.mean().unstack("eval")
        per_seed["gap"] = per_seed["in_dist"] - per_seed["cross_full"]
        short = per_seed.reset_index().groupby(["arch", "cond"]).agg(
            in_dist=("in_dist", _ms), cross=("cross_full", _ms), gap=("gap", _ms),
            n_seeds=("seed", "nunique")).reset_index()
        worst = mat[mat["eval"] == "cross_full"].groupby(["arch", "cond", "train", "test_ds"]).bal_acc.mean()
        if len(worst):
            wi = worst.reset_index().sort_values("bal_acc").groupby(["arch", "cond"]).first()
            short = short.merge(wi.rename(columns={"bal_acc": "worst_pair_bal_acc"}).reset_index(),
                                on=["arch", "cond"], how="left")
            short["worst_pair"] = short.train + "→" + short.test_ds
            short = short.drop(columns=["train", "test_ds"])
        short = short.sort_values(["cond", "arch"], key=lambda s: s.map(cond_order) if s.name == "cond" else s)
        short.to_csv(os.path.join(a.out, "table_short_result.csv"), index=False)
        _latex(short, os.path.join(a.out, "table_short_result.tex"))

    except Exception as e:  # one section failing (e.g. only DANN or LODO runs) must not stop the others
        print('[summarize] section "matrix/short" skipped:', type(e).__name__, e)
    try:
        # (3) gap per pair and condition, and fraction of the base gap closed
        ind = mat[mat["eval"] == "in_dist"][["arch", "cond", "seed", "train", "bal_acc"]].rename(
            columns={"bal_acc": "in_dist"})
        crs = mat[mat["eval"] == "cross_full"][["arch", "cond", "seed", "train", "test_ds", "bal_acc", "auc", "ece"]]
        gp = crs.merge(ind, on=["arch", "cond", "seed", "train"])
        gp["gap"] = gp.in_dist - gp.bal_acc
        bg = gp[gp.cond == "base"].set_index(["arch", "seed", "train", "test_ds"]).gap
        gp["gap_closed"] = [1 - r.gap / bg.get((r.arch, r.seed, r.train, r.test_ds), np.nan)
                            for r in gp.itertuples()]
        GT = gp.groupby(["arch", "train", "test_ds", "cond"]).agg(
            in_dist=("in_dist", _ms), cross=("bal_acc", _ms), gap=("gap", _ms),
            gap_closed=("gap_closed", _ms), cross_auc=("auc", _ms), cross_ece=("ece", _ms),
            n_seeds=("seed", "nunique")).reset_index()
        GT = GT.sort_values(["arch", "train", "test_ds", "cond"],
                            key=lambda s: s.map(cond_order) if s.name == "cond" else s)
        GT.to_csv(os.path.join(a.out, "table_gap_by_condition.csv"), index=False)
        GC = gp.groupby(["arch", "cond", "seed"]).agg(gap=("gap", "mean"), gc=("gap_closed", "mean"),
                                                      cross=("bal_acc", "mean")).reset_index()
        GC = GC.groupby(["arch", "cond"]).agg(mean_cross=("cross", _ms), mean_gap=("gap", _ms),
                                              gap_closed=("gc", _ms)).reset_index()
        GC = GC.sort_values(["arch", "cond"], key=lambda s: s.map(cond_order) if s.name == "cond" else s)
        GC.to_csv(os.path.join(a.out, "table_gap_closed_overall.csv"), index=False)
        _latex(GC, os.path.join(a.out, "table_gap_closed_overall.tex"))

    except Exception as e:  # one section failing (e.g. only DANN or LODO runs) must not stop the others
        print('[summarize] section "gap" skipped:', type(e).__name__, e)
    try:
        # (4) leave-one-dataset-out: trained on all others, tested on the held-out dataset
        lodo = src[src.train.str.contains(r"\+") & (src["eval"] == "cross_full")]
        if len(lodo):
            Lt = lodo.groupby(["arch", "cond", "test_ds"]).bal_acc.apply(_ms).rename("multi_source").reset_index()
            ss = single[single["eval"] == "cross_full"].groupby(["arch", "cond", "test_ds"]).bal_acc
            Lt["single_source_mean"] = [ss.mean().get((r.arch, r.cond, r.test_ds), np.nan)
                                        for r in Lt.itertuples()]
            Lt["single_source_best"] = [ss.max().get((r.arch, r.cond, r.test_ds), np.nan)
                                        for r in Lt.itertuples()]
            up = single[single["eval"] == "in_dist"].groupby(["arch", "cond", "test_ds"]).bal_acc.mean()
            Lt["target_in_dist_upper"] = [up.get((r.arch, r.cond, r.test_ds), np.nan) for r in Lt.itertuples()]
            Lt.round(3).to_csv(os.path.join(a.out, "table_leave_one_dataset_out.csv"), index=False)

    except Exception as e:  # one section failing (e.g. only DANN or LODO runs) must not stop the others
        print('[summarize] section "lodo" skipped:', type(e).__name__, e)
    try:
        # (5) per-sub-class accuracy on other datasets; base models re-scored on harmonised cells
        if not P.empty:
            X = P[(P.method == "src") & (P["eval"] == "cross_full") & ~P.train.str.contains(r"\+")].copy()
            X["correct"] = ((X.logit1 > X.logit0).astype(int) == X.label).astype(float)
            per = X.groupby(["arch", "cond", "train", "test_ds", "seed", "subclass"]).correct.mean().reset_index()
            per.groupby(["arch", "train", "test_ds", "subclass", "cond"]).correct.apply(_ms).unstack(
                "cond").reset_index().to_csv(os.path.join(a.out, "table_subclass_accuracy_cross.csv"), index=False)
            B = P[(P.cond == "base") & (P.method == "src") & P["eval"].isin(["in_dist", "cross_full"])
                  & ~P.train.str.contains(r"\+")]
            recs = []
            for (rid, ev, td), d in B.groupby(["run_id", "eval", "test_ds"]):
                for subset, dd in [("all_cells", d), ("harmonised_cells", d[~d.subclass.isin(DROPPED_IN_HARM)])]:
                    if dd.label.nunique() < 2:
                        continue
                    mt = metrics_from_logits(dd.label.values, dd[["logit0", "logit1"]].values)
                    recs.append(dict(arch=d.arch.iloc[0], train=d.train.iloc[0], test_ds=td,
                                     eval=ev, subset=subset, bal_acc=mt["bal_acc"]))
            if recs:
                H = pd.DataFrame(recs).groupby(["arch", "train", "test_ds", "eval", "subset"]).bal_acc.apply(
                    _ms).unstack("subset").reset_index()
                H.to_csv(os.path.join(a.out, "table_base_on_harmonised_cells.csv"), index=False)

    except Exception as e:  # one section failing (e.g. only DANN or LODO runs) must not stop the others
        print('[summarize] section "subclass" skipped:', type(e).__name__, e)
    try:
        # (5b) calibration in- and cross-dataset: ECE and Brier, raw and temperature-scaled
        Me = mat.groupby(["arch", "cond", "train", "test_ds"]).ece.apply(_ms).unstack("test_ds")
        Me.reindex(columns=dsets).reset_index().to_csv(os.path.join(a.out, "table_matrix_ece.csv"), index=False)
        if not P.empty:
            C = P[(P.method == "src") & P["eval"].isin(["in_dist", "cross_full"]) & ~P.train.str.contains(r"\+")]
            recs = []
            for (rid, ev, td), d in C.groupby(["run_id", "eval", "test_ds"]):
                z = d[["logit0", "logit1"]].values.astype(float)
                y = d.label.values.astype(int)
                T = float(d.T_src.iloc[0])
                p_raw, p_ts = softmax_np(z)[:, 1], softmax_np(z / T)[:, 1]
                recs.append(dict(arch=d.arch.iloc[0], cond=d.cond.iloc[0], train=d.train.iloc[0],
                                 seed=d.seed.iloc[0], test_ds=td, regime="in" if ev == "in_dist" else "cross",
                                 T=T, ece_raw=ece_score(y, p_raw), ece_ts=ece_score(y, p_ts),
                                 brier_raw=float(np.mean((p_raw - y) ** 2)),
                                 brier_ts=float(np.mean((p_ts - y) ** 2)),
                                 mean_conf=float(np.mean(np.maximum(p_raw, 1 - p_raw))),
                                 acc=float(np.mean((p_raw >= 0.5) == y))))
            if recs:
                CR = pd.DataFrame(recs)
                CR["ts_worse"] = CR.ece_ts > CR.ece_raw
                CR["overconfidence"] = CR.mean_conf - CR.acc
                CR.to_csv(os.path.join(a.out, "calibration_runs.csv"), index=False)
                CT = CR.groupby(["arch", "cond", "train", "test_ds", "regime"]).agg(
                    ece_raw=("ece_raw", _ms), ece_ts=("ece_ts", _ms), brier_raw=("brier_raw", _ms),
                    overconfidence=("overconfidence", _ms), T_median=("T", "median"),
                    ts_worse_runs=("ts_worse", "sum"), runs=("ts_worse", "size")).reset_index()
                CT.to_csv(os.path.join(a.out, "table_calibration.csv"), index=False)
                CS = CR.groupby(["arch", "cond", "regime"]).agg(
                    ece_raw=("ece_raw", "median"), ece_ts=("ece_ts", "median"),
                    ts_worse=("ts_worse", "sum"), runs=("ts_worse", "size")).reset_index()
                CS.round(3).to_csv(os.path.join(a.out, "table_calibration_summary.csv"), index=False)
                print("\n== calibration (median ECE over runs/pairs)\n", CS.round(3).to_string(index=False))
                # reliability diagrams: base condition, first seed, rows = architecture, cols = training set
                B = C[(C.cond == "base")]
                if len(B):
                    s0 = B.seed.min()
                    B = B[B.seed == s0]
                    archs = sorted(B.arch.unique())
                    trains = [d for d in dsets if d in set(B.train)]
                    fig, axs = plt.subplots(len(archs), len(trains), figsize=(2.5 * len(trains), 2.4 * len(archs)),
                                            squeeze=False, sharex=True, sharey=True)
                    pal = {ds: c for ds, c in zip(dsets, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#6250d6"])}
                    edges = np.linspace(0.5, 1.0, 11)
                    for i, arch in enumerate(archs):
                        for j, tr in enumerate(trains):
                            ax = axs[i, j]
                            ax.plot([0.5, 1], [0.5, 1], ls="--", color="#b5b4ae", lw=0.8)
                            for td, d in B[(B.arch == arch) & (B.train == tr)].groupby("test_ds"):
                                pr = softmax_np(d[["logit0", "logit1"]].values.astype(float))[:, 1]
                                conf, acc = np.maximum(pr, 1 - pr), ((pr >= 0.5) == d.label.values)
                                xs, ys = [], []
                                for lo, hi in zip(edges[:-1], edges[1:]):
                                    mk = (conf > lo) & (conf <= hi)
                                    if mk.sum() >= 5:
                                        xs.append(conf[mk].mean())
                                        ys.append(acc[mk].mean())
                                ax.plot(xs, ys, marker="o", ms=3, lw=1.6 if td == tr else 1.0,
                                        color=pal.get(td, "#52514e"), label=td + (" (in)" if td == tr else ""))
                            ax.set_title(f"{arch} | trained on {tr}", fontsize=7)
                            ax.set_xlim(0.5, 1.0)
                            ax.set_ylim(0.0, 1.0)
                            if i == len(archs) - 1:
                                ax.set_xlabel("confidence", fontsize=7)
                            if j == 0:
                                ax.set_ylabel("accuracy", fontsize=7)
                            ax.legend(fontsize=5, loc="lower right")
                    fig.tight_layout()
                    fig.savefig(os.path.join(a.out, "fig_reliability_base.png"), dpi=200)
                    plt.close(fig)

    except Exception as e:  # one section failing (e.g. only DANN or LODO runs) must not stop the others
        print('[summarize] section "calibration" skipped:', type(e).__name__, e)
    try:
        # (6a) threshold-only adaptation of the unchanged source model (post hoc, from saved logits)
        if not P.empty and os.path.exists(a.manifest):
            man = pd.read_csv(a.manifest, keep_default_na=False)
            shots = sorted({int(m[2:]) for m in raw.method.unique() if m.startswith("ft") and m[2:].isdigit()}) \
                or [5, 10, 25, 50]
            TH = threshold_posthoc(P, man, shots)
            if len(TH):
                TH.to_csv(os.path.join(a.out, "threshold_runs.csv"), index=False)
                base_tt = TH[TH.method == "thr_base"].groupby(["arch", "cond", "train", "test_ds"]).bal_acc.mean()
                raw = pd.concat([raw, TH[TH.method != "thr_base"]], ignore_index=True)
                orc = TH[TH.method == "thr_oracle"].groupby(["arch", "cond", "train", "test_ds"]).bal_acc.mean()
                upd = single[single["eval"] == "in_dist"].groupby(["arch", "cond", "train"]).bal_acc.mean()
                D = pd.DataFrame({"base": base_tt, "best_threshold": orc}).dropna().reset_index()
                D["target_trained"] = [upd.get((r.arch, r.cond, r.test_ds), np.nan) for r in D.itertuples()]
                D["gap"] = D.target_trained - D.base
                D["threshold_part"] = D.best_threshold - D.base
                D["representation_part"] = D.target_trained - D.best_threshold
                D["threshold_share_of_gap"] = D.threshold_part / D.gap
                D.round(3).to_csv(os.path.join(a.out, "table_gap_decomposition.csv"), index=False)
                print("\n== how much of the gap is just the decision threshold?\n",
                      D.groupby(["arch", "cond"])[["gap", "threshold_part", "representation_part",
                                                    "threshold_share_of_gap"]].mean().round(3).to_string())
        elif not P.empty:
            print(f"[summarize] {a.manifest} not found: threshold analysis skipped")

    except Exception as e:  # one section failing (e.g. only DANN or LODO runs) must not stop the others
        print('[summarize] section "threshold" skipped:', type(e).__name__, e)
    try:
        # (6) adaptation (AdaBN, few-shot, threshold-only) on target test splits
        tt = raw[(raw["eval"] == "cross_tgtTest") & ~raw.train.str.contains(r"\+")]
        if len(tt):
            A = tt.groupby(["arch", "cond", "train", "test_ds", "method"]).bal_acc.apply(_ms).unstack("method")
            A = A[sorted(A.columns, key=_mkey)].reset_index()
            up = single[single["eval"] == "in_dist"].groupby(["arch", "cond", "train"]).bal_acc.mean()
            A["target_in_dist_upper"] = [up.get((r.arch, r.cond, r.test_ds), np.nan) for r in A.itertuples()]
            A.to_csv(os.path.join(a.out, "table_adaptation.csv"), index=False)
            # average few-shot curve over all train->target pairs
            shots = sorted({int(m[2:]) for m in tt.method.unique() if m.startswith("ft") and m[2:].isdigit()})
            if shots:
                for cond in sorted(set(tt.cond), key=lambda c: cond_order.get(c, 99)):
                    d = tt[tt.cond == cond]
                    if not any(m.startswith("ft") for m in d.method.unique()):
                        continue
                    archs = sorted(d.arch.unique())
                    fig, axs = plt.subplots(1, len(archs), figsize=(3.3 * len(archs), 2.7), squeeze=False)
                    for ax, arch in zip(axs[0], archs):
                        dd = d[d.arch == arch]
                        pair_mean = lambda meth: dd[dd.method == meth].groupby(["train", "test_ds"]).bal_acc.mean().mean()
                        for pref, lab, col in [("ft", "from source model", "#2a78d6"),
                                               ("ftscratch", "from ImageNet only", "#52514e")]:
                            ys = [pair_mean(f"{pref}{k}") for k in shots]
                            if not np.all(np.isnan(ys)):
                                ax.plot(shots, ys, marker="o", color=col, label=lab)
                        ys = [pair_mean(f"thr{k}") for k in shots]
                        if not np.all(np.isnan(ys)):
                            ax.plot(shots, ys, marker="s", color="#e87ba4",
                                    label="threshold only (model unchanged)")
                        ax.axhline(pair_mean("src"), ls=":", color="#eb6834", label="no target labels")
                        if "thr_oracle" in set(dd.method):
                            ax.axhline(pair_mean("thr_oracle"), ls=(0, (1, 3)), color="#52514e",
                                       label="best possible threshold")
                        if "adabn" in set(dd.method):
                            ax.axhline(pair_mean("adabn"), ls="-.", color="#eda100", label="AdaBN (unlabelled)")
                        ups = [up.get((arch, cond, t), np.nan) for t in sorted(set(dd.test_ds))]
                        if not np.all(np.isnan(ups)):
                            ax.axhline(np.nanmean(ups), ls="--", color="#1baf7a", label="trained on target")
                        ax.set_xscale("log")
                        ax.set_xticks(shots)
                        ax.set_xticklabels(shots)
                        ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
                        ax.set_xlabel("labelled target cells per class")
                        ax.set_title(f"{arch} ({cond}), mean over pairs", fontsize=8)
                    axs[0][0].set_ylabel("Target-test balanced acc.")
                    hl = {}
                    for ax in axs[0]:
                        for h, l in zip(*ax.get_legend_handles_labels()):
                            hl.setdefault(l, h)
                    fig.legend(list(hl.values()), list(hl.keys()), loc="lower center", ncol=4, fontsize=6.5,
                               frameon=False, bbox_to_anchor=(0.5, 0.0))
                    fig.tight_layout(rect=(0, 0.12, 1, 1))
                    fig.savefig(os.path.join(a.out, f"fig_fewshot_{cond}.png"), dpi=200)
                    plt.close(fig)
    except Exception as e:  # one section failing (e.g. only DANN or LODO runs) must not stop the others
        print('[summarize] section "adaptation" skipped:', type(e).__name__, e)
    try:
        ret = raw[(raw.method != "src") & (raw["eval"] == "in_dist")]
        if len(ret):
            ret.groupby(["arch", "cond", "train", "method"]).bal_acc.apply(_ms).unstack("method").reset_index().to_csv(
                os.path.join(a.out, "table_adaptation_source_retention.csv"), index=False)
    except Exception as e:  # one section failing (e.g. only DANN or LODO runs) must not stop the others
        print('[summarize] section "retention" skipped:', type(e).__name__, e)

    try:
        # figure: cross-dataset heat maps (balanced accuracy), one panel per architecture
        for cond in sorted(set(mat.cond), key=lambda c: cond_order.get(c, 99)):
            mc = mat[mat.cond == cond]
            archs = sorted(mc.arch.unique())
            trains = [d for d in dsets if d in set(mc.train)]
            fig, axs = plt.subplots(1, len(archs), figsize=(2.9 * len(archs) + 0.6, 2.8), squeeze=False)
            for ai, (ax, arch) in enumerate(zip(axs[0], archs)):
                g = mc[mc.arch == arch].groupby(["train", "test_ds"]).bal_acc.mean().unstack("test_ds")
                g = g.reindex(index=trains, columns=dsets)
                im = ax.imshow(g.values, vmin=0.5, vmax=1.0, cmap="Blues")
                for i in range(g.shape[0]):
                    for j in range(g.shape[1]):
                        v = g.values[i, j]
                        if np.isfinite(v):
                            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                                    color="white" if v > 0.8 else "#0b0b0b")
                ax.set_xticks(range(len(dsets)))
                ax.set_xticklabels(dsets, rotation=40, ha="right", fontsize=7)
                ax.set_yticks(range(len(trains)))
                ax.set_yticklabels(trains if ai == 0 else [], fontsize=7)
                ax.set_title(arch, fontsize=8)
                ax.set_xlabel("test dataset", fontsize=7)
            axs[0][0].set_ylabel("trained on", fontsize=7)
            fig.colorbar(im, ax=axs[0].tolist(), shrink=0.8, label="balanced accuracy")
            fig.savefig(os.path.join(a.out, f"fig_matrix_{cond}.png"), dpi=200, bbox_inches="tight")
            plt.close(fig)
    except Exception as e:  # one section failing (e.g. only DANN or LODO runs) must not stop the others
        print('[summarize] section "heatmaps" skipped:', type(e).__name__, e)
    try:
        # figure: mean cross-dataset balanced accuracy per condition
        if gp.cond.nunique() > 1:
            archs = sorted(gp.arch.unique())
            conds = [c for c in CONDS if c in set(gp.cond)]
            fig, ax = plt.subplots(figsize=(1.0 + 0.9 * len(conds), 2.8))
            w = 0.8 / len(archs)
            palette = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]
            for i, arch in enumerate(archs):
                per = gp[gp.arch == arch].groupby(["cond", "seed"]).bal_acc.mean().reset_index()
                mu = [per[per.cond == c].bal_acc.mean() for c in conds]
                sd = [per[per.cond == c].bal_acc.std(ddof=1) for c in conds]
                ax.bar(np.arange(len(conds)) + (i - (len(archs) - 1) / 2) * w, mu, w * 0.92, yerr=sd,
                       capsize=2, color=palette[i % len(palette)], label=arch)
            ax.set_xticks(range(len(conds)))
            ax.set_xticklabels(conds, rotation=30, ha="right", fontsize=7)
            ax.set_ylim(0.5, 1.0)
            ax.set_ylabel("mean cross-dataset bal. acc.")
            ax.legend(fontsize=6, ncol=2)
            fig.tight_layout()
            fig.savefig(os.path.join(a.out, "fig_conditions.png"), dpi=200)
            plt.close(fig)
        pd.set_option("display.width", 220)
    except Exception as e:  # one section failing (e.g. only DANN or LODO runs) must not stop the others
        print('[summarize] section "conditions fig" skipped:', type(e).__name__, e)
    try:
        print("== short result\n", short.to_string(index=False))
        print("\n== cross-dataset matrix (balanced accuracy)\n", M.to_string(index=False))
    except Exception as e:  # one section failing (e.g. only DANN or LODO runs) must not stop the others
        print('[summarize] section "print" skipped:', type(e).__name__, e)
    print(f"\n[summarize] tables and figures written to {a.out}")


# ----------------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    p = sub.add_parser("check", help="compare a downloaded copy with the official release")
    p.add_argument("--root", required=True)
    p.add_argument("--name", choices=list(OFFICIAL), required=True)
    p = sub.add_parser("crops", help="cut cells around nucleus points / YOLO boxes")
    p.add_argument("--images", required=True, help="folder with the full-field images")
    p.add_argument("--csv", help="point annotation CSV, or a folder of CSVs")
    p.add_argument("--yolo", action="store_true", help="annotations are YOLO .txt boxes")
    p.add_argument("--labels", help="folder with YOLO .txt files (default: search --images)")
    p.add_argument("--classes", help="classes.txt or data.yaml (default: search)")
    p.add_argument("--out", required=True)
    p.add_argument("--size", type=int, default=128, help="crop side in pixels")
    p.add_argument("--col_image")
    p.add_argument("--col_x")
    p.add_argument("--col_y")
    p.add_argument("--col_label")
    p.add_argument("--col_cell")
    p = sub.add_parser("manifest")
    p.add_argument("--sipakmed_root")
    p.add_argument("--herlev_root")
    p.add_argument("--crops", action="append", help="NAME=crops_dir (repeatable), e.g. CRIC=crops/CRIC")
    p.add_argument("--out", default="manifest.csv")
    p.add_argument("--seed", type=int, default=0)
    p = sub.add_parser("probe")
    p.add_argument("--manifest", default="manifest.csv")
    p.add_argument("--out", default="results")
    p = sub.add_parser("ruler")
    p.add_argument("--manifest", default="manifest.csv")
    p.add_argument("--out", default="results")
    p.add_argument("--reference", default="SIPaKMeD")
    p.add_argument("--max_cells", type=int, default=600)
    p = sub.add_parser("train")
    p.add_argument("--manifest", default="manifest.csv")
    p.add_argument("--ruler", default="results/ruler.json")
    p.add_argument("--datasets", default="", help="restrict to these datasets (comma list)")
    p.add_argument("--arch", choices=list(TIMM_NAMES) + list(HYBRIDS) + ["custom"], required=True)
    p.add_argument("--custom_module", default="custom_model.py",
                   help="file defining build_model(num_classes) for --arch custom")
    p.add_argument("--train_ds", required=True,
                   help="dataset name, or several comma-separated (leave-one-dataset-out)")
    p.add_argument("--cond", choices=list(CONDS), required=True)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--epochs", type=int, default=10)
    p.add_argument("--bs", type=int, default=32)
    p.add_argument("--lr", type=float, default=None,
                   help="default 3e-5 for transformers (vit_b16, swin_t), 1e-4 otherwise")
    p.add_argument("--workers", type=int, default=2)
    p.add_argument("--meta", choices=["abnormal", "normal", "drop"])
    p.add_argument("--columnar", choices=["normal", "drop"])
    p.add_argument("--ascus", choices=["abnormal", "drop"])
    p.add_argument("--adabn", action="store_true")
    p.add_argument("--fewshot", default="", help="e.g. 5,10,25,50")
    p.add_argument("--ft_scratch", action="store_true")
    p.add_argument("--adapt_to", default="all", help="'all' or comma list of target datasets")
    p.add_argument("--dann_target", default="", help="train with DANN towards this (unlabelled) target dataset")
    p.add_argument("--ft_epochs", type=int, default=20)
    p.add_argument("--ft_lr", type=float, default=None, help="default: same as --lr")
    p.add_argument("--limit", type=int, default=0, help="smoke test: cells per dataset/split")
    p.add_argument("--out", default="runs")
    p.add_argument("--overwrite", action="store_true")
    p.add_argument("--save_weights", action="store_true", help="save the source model (.pt) for xai")
    p = sub.add_parser("xai", help="quantitative Grad-CAM vs nucleus masks on every dataset")
    p.add_argument("--manifest", default="manifest.csv")
    p.add_argument("--ruler", default="results/ruler.json")
    p.add_argument("--runs", default="runs_xai", help="folder with <arch>_<train>_<cond>_s<seed>.pt")
    p.add_argument("--arch", choices=list(TIMM_NAMES) + list(HYBRIDS) + ["custom"], required=True)
    p.add_argument("--custom_module", default="custom_model.py")
    p.add_argument("--cond", choices=list(CONDS), default="base")
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--max_images", type=int, default=400, help="test cells per dataset")
    p.add_argument("--bs", type=int, default=32)
    p.add_argument("--out", default="results_xai")
    p = sub.add_parser("summarize")
    p.add_argument("--runs", default="runs")
    p.add_argument("--out", default="results")
    p.add_argument("--no_preds", action="store_true", help="skip per-image tables (faster)")
    p.add_argument("--manifest", default="manifest.csv",
                   help="needed for the threshold analysis (target train/test split)")
    a = ap.parse_args()
    if a.cmd == "crops" and not a.yolo and not a.csv:
        sys.exit("[crops] give --csv (point annotations) or --yolo")
    {"check": cmd_check, "crops": cmd_crops, "manifest": cmd_manifest, "probe": cmd_probe, "ruler": cmd_ruler,
     "train": cmd_train, "summarize": cmd_summarize, "xai": cmd_xai}[a.cmd](a)


if __name__ == "__main__":
    main()

## 1. Settings check, dataset paths, packages

In [ ]:
# ---- automatic path detection (you only need to edit the OVERRIDE lines if a guess is wrong)
import os, csv
INPUT = "/kaggle/input"
IMG = (".bmp", ".png", ".jpg", ".jpeg", ".tif", ".tiff")
WALK = [(d, s, f) for d, s, f in os.walk(INPUT)]

def n_images(root):
    return sum(1 for d, s, f in WALK if d == root or d.startswith(root + os.sep) for x in f if x.lower().endswith(IMG))

def top_of(d):
    # the dataset's own top folder: /kaggle/input/<slug> (or .../datasets/<owner>/<slug>)
    parts = os.path.relpath(d, INPUT).split(os.sep)
    keep = 3 if parts[0] == "datasets" and len(parts) >= 3 else 1
    return os.path.join(INPUT, *parts[:keep])

def lift(d, min_images):
    # walk up from d until the folder holds at least min_images images, never leaving the dataset
    top = top_of(d)
    while d != top and n_images(d) < min_images:
        d = os.path.dirname(d)
    return d

def header(p):
    try:
        with open(p, newline="", errors="ignore") as fh:
            return [c.strip().lower() for c in next(csv.reader(fh))]
    except Exception:
        return []

SIPAK = HERLEV = CRIC_ROOT = CRIC_CSV = APC_ROOT = ""
for d, s, f in WALK:
    low = [x.lower() for x in s]
    if not SIPAK and any(x.startswith("im_superficial") for x in low) and any(x.startswith("im_dyskerat") for x in low):
        SIPAK = d
    if not HERLEV and any("dysplas" in x for x in low) and any("superfic" in x or "columnar" in x for x in low):
        HERLEV = d
# use each dataset's top folder (catches copies that are split into train/ and test/ sub-folders)
if SIPAK and HERLEV and top_of(SIPAK) != top_of(HERLEV):
    SIPAK, HERLEV = top_of(SIPAK), top_of(HERLEV)
cric_csvs, apc_dirs = [], []
for d, s, f in WALK:
    for x in f:
        p = os.path.join(d, x)
        if x.lower().endswith(".csv"):
            h = header(p)
            if any("bethesda" in c for c in h) and any("nucleus_x" in c or c == "x" for c in h):
                cric_csvs.append(p)
        if x.lower() in ("classes.txt", "data.yaml"):
            apc_dirs.append(d)
cric_csvs.sort(key=lambda p: (0 if "cric" in p.lower() else 1, len(p)))
apc_dirs.sort(key=lambda p: (0 if "apc" in p.lower() else 1, len(p)))
if cric_csvs:
    CRIC_CSV = cric_csvs[0]
    CRIC_ROOT = lift(os.path.dirname(CRIC_CSV), 100)
if apc_dirs:
    APC_ROOT = lift(apc_dirs[0], 100)

# ---- OVERRIDE here only if a printed guess below is wrong ----
# SIPAK = "..."; HERLEV = "..."; CRIC_ROOT = "..."; CRIC_CSV = "..."; APC_ROOT = ""   ("" skips APCData)

CRIC_IMAGES, APC_IMAGES, APC_LABELS = CRIC_ROOT, APC_ROOT, APC_ROOT
for name, val in [("SIPAK", SIPAK), ("HERLEV", HERLEV), ("CRIC_ROOT", CRIC_ROOT), ("CRIC_CSV", CRIC_CSV), ("APC_ROOT", APC_ROOT)]:
    extra = f"  ({n_images(val)} images inside)" if val and os.path.isdir(val) else ""
    print(f"{name:10s} = {val or 'NOT FOUND'}{extra}")
assert SIPAK and HERLEV and CRIC_ROOT and CRIC_CSV, "a dataset was not found: set it by hand in the OVERRIDE line"
CRIC_CROP, APC_CROP = 224, 256
os.chdir("/kaggle/working")
import glob, subprocess, sys, importlib.util
import pandas as pd
from IPython.display import Image as IPImage, display
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
# Kaggle already has timm and scikit-image. Install only if missing, WITHOUT dependencies, so that
# pip can never replace PyTorch (that is what broke the previous run).
missing = [pkg for pkg, mod in [("timm", "timm"), ("scikit-image", "skimage")] if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps"] + missing, check=True)
# GPU check without importing torch in this notebook (training runs in separate processes)
gpu_list = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
N_GPUS = len([g for g in gpu_list if g.startswith("GPU")])
assert N_GPUS > 0, "NO GPU! Settings -> Accelerator -> GPU T4 x2, then Save Version again"
print(f"{N_GPUS} GPU(s):", gpu_list)
check = subprocess.run([sys.executable, "-c", "import torch, timm, skimage; print('torch', torch.__version__, '| timm', timm.__version__, '| cuda', torch.cuda.is_available())"],
                       capture_output=True, text=True)
print(check.stdout, check.stderr[-1500:])
assert check.returncode == 0 and "cuda True" in check.stdout, "PyTorch/timm check failed: see the message above"

## 2. Cell crops, manifest (identical in every notebook: everything is seeded), ruler

In [ ]:
!python gapfix.py crops --images "$CRIC_IMAGES" --csv "$CRIC_CSV" --out crops/CRIC --size $CRIC_CROP
assert os.path.exists("crops/CRIC/crops_index.csv"), "CRIC crops failed: read the message above"
!python gapfix.py crops --images "$APC_IMAGES" --yolo --labels "$APC_LABELS" --out crops/APCData --size $APC_CROP
assert os.path.exists("crops/APCData/crops_index.csv"), "APCData crops failed: read the message above"
!python gapfix.py manifest --sipakmed_root "$SIPAK" --herlev_root "$HERLEV" --crops CRIC=crops/CRIC --crops APCData=crops/APCData --out manifest.csv
assert os.path.exists("manifest.csv"), "manifest failed: read the message above"
!python gapfix.py ruler --manifest manifest.csv --out results
assert os.path.exists("results/ruler.json"), "ruler failed: read the message above"
display(IPImage("results/ruler_check.png"))

## 3. Job runner
Both GPUs in parallel. After **every** finished run: (a) a `RESULT` line with all its numbers is printed here, so even a notebook that stops early still contains the results; (b) `results_S6.zip` in /kaggle/working is updated, so it can be downloaded at any moment (right panel → Output). Runs already present in an input (a previous `results_*.zip` or `runs` folder added with **Add Input**) are copied and skipped, so a stopped notebook continues where it stopped.

In [ ]:
import subprocess, time, itertools, json, gzip, base64, zipfile, shutil
DATASETS = sorted(pd.read_csv("manifest.csv").dataset.unique()); print(DATASETS)
T_START = globals().get("T_START", time.time())
BUDGET_H = 3.0
TAG = "results_S6"
os.makedirs("runs", exist_ok=True); os.makedirs("logs", exist_ok=True)
def run_id(arch, train, cond, seed, dann=""):
    return f"{arch}_{'+'.join(train.split(','))}_{cond}" + (f"_dann-{dann}" if dann else "") + f"_s{seed}"

# ---- resume: copy finished runs from any input (previous results zip or runs folder)
n_imp = 0
for root, dirs, files in os.walk("/kaggle/input"):
    for fn in files:
        fp = os.path.join(root, fn)
        if fn.endswith(".zip") and fn.startswith("results_"):
            with zipfile.ZipFile(fp) as z:
                for mem in z.namelist():
                    b = os.path.basename(mem)
                    if mem.startswith("runs/") and (b.endswith(".json") or b.endswith("_preds.csv.gz")) and not os.path.exists("runs/" + b):
                        open("runs/" + b, "wb").write(z.read(mem)); n_imp += b.endswith(".json")
        elif os.path.basename(root) == "runs" and (fn.endswith(".json") or fn.endswith("_preds.csv.gz")) and not os.path.exists("runs/" + fn):
            shutil.copy(fp, "runs/" + fn); n_imp += fn.endswith(".json")
print(f"resume: {n_imp} finished run(s) copied from inputs" if n_imp else "resume: nothing to copy (fresh start)")

def _add_to_zip(rid):
    with zipfile.ZipFile(TAG + ".zip", "a", zipfile.ZIP_DEFLATED) as z:
        have = set(z.namelist())
        for f in ["manifest.csv", "results/ruler.json", f"runs/{rid}.json", f"runs/{rid}_preds.csv.gz", f"logs/{rid}.txt"]:
            if os.path.exists(f) and f not in have:
                z.write(f)

def _report(rid, j):
    d = json.load(open(f"runs/{rid}.json"))
    print("RESULT", rid, base64.b64encode(gzip.compress(json.dumps(d, separators=(",", ":")).encode())).decode(), flush=True)
    raw = [r for r in d["rows"] if r.get("calib") == "raw"]
    ind = [r["bal_acc"] for r in raw if r["eval"] == "in_dist" and r["method"] in ("src", "dann")]
    crs = [r["bal_acc"] for r in raw if r["eval"] == "cross_full" and r["method"] in ("src", "dann")]
    f = lambda v: f"{sum(v)/len(v):.3f}" if v else "-"
    return f"own-test BA {f(ind)} | other-datasets BA {f(crs)}"

def run_jobs(jobs, out="runs"):
    rid = lambda j: run_id(j["arch"], j["train"], j["cond"], j["seed"], j.get("dann", ""))
    for r in [rid(j) for j in jobs if os.path.exists(f"{out}/{rid(j)}.json")]:
        _add_to_zip(r)
    jobs = [j for j in jobs if not os.path.exists(f"{out}/{rid(j)}.json")]
    gpus = list(range(N_GPUS)); running = {}
    print(f"{len(jobs)} runs to do on {len(gpus)} GPU(s)", flush=True)
    while jobs or running:
        if jobs and (time.time() - T_START) / 3600 > BUDGET_H:
            print(f"TIME BUDGET REACHED: {len(jobs)} runs not started", flush=True); jobs = []
        for g in gpus:
            if g not in running and jobs:
                j = jobs.pop(0)
                cmd = ["python", "gapfix.py", "train", "--arch", j["arch"], "--train_ds", j["train"],
                       "--cond", j["cond"], "--seed", str(j["seed"]), "--out", out] + j.get("extra", []) + \
                      (["--dann_target", j["dann"]] if j.get("dann") else [])
                log = open(f"logs/{rid(j)}.txt", "w")
                running[g] = (subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT,
                              env={**os.environ, "CUDA_VISIBLE_DEVICES": str(g)}), j)
        time.sleep(10)
        for g, (p, j) in list(running.items()):
            if p.poll() is not None:
                r = rid(j); t = f"[{(time.time()-T_START)/3600:.2f} h]"
                if p.returncode == 0 and os.path.exists(f"{out}/{r}.json"):
                    msg = _report(r, j); _add_to_zip(r)
                    print(f"{t} ok {r} | {msg}", flush=True)
                else:
                    print(f"{t} FAILED rc={p.returncode} {r}; last lines of its log:", flush=True)
                    print("".join(open(f"logs/{r}.txt").readlines()[-25:]), flush=True)
                del running[g]

## 4. Runs

In [ ]:
run_jobs([dict(arch="vit_b16", train=d, cond="base", seed=0, extra=["--save_weights"]) for d in DATASETS])

## 5. Grad-CAM

In [ ]:
!python gapfix.py xai --arch vit_b16 --cond base --runs runs --out results_xai
for f in sorted(glob.glob("results_xai/fig_xai_*.png")): display(IPImage(f))

## Summary and download
Download `results_S6.zip` (version **Output** tab, or right panel → Output) and send it back.

In [ ]:
!python gapfix.py summarize --runs runs --out results --manifest manifest.csv
for f in ["table_short_result", "table_matrix_balacc", "table_calibration_summary", "table_gap_decomposition",
          "table_gap_closed_overall", "table_leave_one_dataset_out", "table_adaptation"]:
    if os.path.exists(f"results/{f}.csv"): print("\n==", f); display(pd.read_csv(f"results/{f}.csv"))
!zip -qr results_S6.zip results runs manifest.csv logs crops/CRIC/crops_check.png crops/APCData/crops_check.png results_xai -x "*.pt"
!ls -lh results_S6.zip